# BRIDGE / TCH-Net — temporal re-split only (3 seeds)

The full revised notebook hit Kaggle's 12-hour limit during the third seed of the temporal re-split (Table 16). This notebook runs **only** that experiment. It keeps the setup cells from `Revised_Version_of_BRIDGE_and_TCH_Net.ipynb` unchanged: imports, vocabulary, `Config`, loaders, data split and windowing, models, and training infrastructure. Then it runs the temporal-split cell for all three seeds `Config.FAST_SEEDS = [42, 123, 456]`.

**Run on Kaggle:** attach the same four datasets, select the GPU T4 accelerator, and *Run All*. Expected time is roughly 30–45 minutes.

The result is written to `tch_net_v2_results/temporal.json` and printed in the last cell.


In [1]:
import os, glob, json, time, warnings, copy, gc, itertools, math
warnings.filterwarnings('ignore')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython()
    import matplotlib.pyplot as plt
except NameError:
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import matplotlib.ticker as ticker

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    f1_score, accuracy_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, auc, matthews_corrcoef)
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from scipy import stats as scipy_stats

try:
    from xgboost import XGBClassifier; XGB_OK = True
except ImportError:
    XGB_OK = False; print("XGBoost unavailable — fallback to GradientBoosting")

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True      # fastest CUDA kernels
    torch.backends.cudnn.deterministic = False  # benchmark requires non-deterministic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")
print(f"CUDA: {torch.cuda.is_available()} | XGBoost: {XGB_OK}")


PyTorch 2.11.0+cu128 | Device: cuda
CUDA: True | XGBoost: True


In [2]:
# ── 46 canonical features ─────────────────────────────────────────────────
SEMANTIC_FEATURES = [
    # ── Flow-level counts & rates (T-branch primary) ─────────────────────
    "flow_duration",        # 0  total flow duration
    "pkt_count_fwd",        # 1  total forward packets
    "pkt_count_bwd",        # 2  total backward packets
    "byte_count_fwd",       # 3  total forward bytes
    "byte_count_bwd",       # 4  total backward bytes
    "pkt_rate",             # 5  packets per second (flow level)
    "byte_rate",            # 6  bytes per second
    "fwd_pkt_rate",         # 7  forward packets per second
    "bwd_pkt_rate",         # 8  backward packets per second
    "fwd_byte_rate",        # 9  forward bytes per second
    "bwd_byte_rate",        # 10 backward bytes per second
    "pkt_count_total",      # 11 total packets (fwd + bwd)
    "byte_count_total",     # 12 total bytes
    "fwd_pkt_len_total",    # 13 total length of fwd packets
    "bwd_pkt_len_total",    # 14 total length of bwd packets
    "subflow_fwd_pkts",     # 15 subflow forward packets
    "subflow_bwd_pkts",     # 16 subflow backward packets
    # ── Packet size statistics (H-branch primary) ────────────────────────
    "pkt_len_min",          # 17
    "pkt_len_max",          # 18
    "pkt_len_mean",         # 19
    "pkt_len_std",          # 20
    "pkt_len_var",          # 21
    "fwd_pkt_len_min",      # 22
    "fwd_pkt_len_max",      # 23
    "fwd_pkt_len_mean",     # 24
    "fwd_pkt_len_std",      # 25
    "bwd_pkt_len_min",      # 26
    "bwd_pkt_len_max",      # 27
    "bwd_pkt_len_mean",     # 28
    "bwd_pkt_len_std",      # 29
    # ── IAT statistics (T + H) ───────────────────────────────────────────
    "iat_mean",             # 30
    "iat_std",              # 31
    "iat_max",              # 32
    "iat_min",              # 33
    "fwd_iat_mean",         # 34
    "fwd_iat_std",          # 35
    "bwd_iat_mean",         # 36
    "bwd_iat_std",          # 37
    # ── TCP flags (H-branch) ─────────────────────────────────────────────
    "flag_syn",             # 38
    "flag_ack",             # 39
    "flag_fin",             # 40
    "flag_rst",             # 41
    "flag_psh",             # 42
    "flag_urg",             # 43
    # ── Header / window ──────────────────────────────────────────────────
    "fwd_header_len",       # 44
    "init_win_fwd",         # 45
]
N_SEM = len(SEMANTIC_FEATURES)
SEM_IDX = {name: i for i, name in enumerate(SEMANTIC_FEATURES)}
print(f"Canonical vocabulary: {N_SEM} features")

# ── Per-dataset alias maps ───────────────────────────────────────────────
# Key = canonical name. Value = list of possible column names (case-insensitive).
# ONLY genuinely equivalent features are mapped. No fabricated proxies.
# Short aliases (<5 chars) NEVER matched via substring — only exact match.
#
# ── DATASET RESET (v7) ────────────────────────────────────────────────────
# CIC-IoT-DIAD-2024 (Mostafa Ali Ogab, Kaggle) was confirmed BROKEN — every
# file (including attack-named files) carries "NeedManualLabel" in the Label
# column. Genuinely unusable for supervised learning. DROPPED.
# CICIoMT-2024 (Lian Mollick, Kaggle) was confirmed WRONG VARIANT — 45 cols,
# no label column at all (same reduced schema as CIC-IoT-2023). DROPPED.
#
# Replaced with, and verified directly against user-supplied sample CSVs:
#   • CICIDS-2017                          (sample: cicids-2017-friday[-plus].csv)
#   • CIC-BCCC-NRC-TabularIoTAttacks-2024   (sample: cic-bccc-nrc-tabulariotAttack.csv)
#   • CICIoMT-2024 (Train-Test, D. Kilichev) (sample: ciciomt-{train,test}-set.csv)

# ── ALIAS_FLOWMETER ──────────────────────────────────────────────────────
# Shared by CICIDS-2017 AND CIC-BCCC-NRC-TabularIoTAttacks-2024.
# Verified column-for-column against both uploaded samples — they use the
# IDENTICAL CICFlowMeter naming convention (BCCC's updated extractor):
#   "Total Fwd Packet"           (singular, no trailing 's')
#   "Total Bwd packets"          (lowercase 'p', trailing 's')
#   "Total Length of Fwd Packet" (singular)
#   "Total Length of Bwd Packet" (singular)
#   "FWD Init Win Bytes"         (upper-case FWD)
# Old verbose CICIDS MachineLearningCSV.zip names ("Total Fwd Packets",
# "Init_Win_bytes_forward", ...) are kept as fallback aliases in case a
# differently-formatted CICIDS-2017 file is substituted later.
# cic-bccc-nrc-tabulariotAttack.csv additionally carries an explicit
# "Attack Name" string column (e.g. "Benign Traffic") used for labeling —
# see load_cic_bccc_nrc() below — it is NOT part of the feature vocabulary.
ALIAS_FLOWMETER = {
    "flow_duration":     ["Flow Duration"],
    "pkt_count_fwd":     ["Total Fwd Packet",              "Total Fwd Packets",
                          "Tot Fwd Pkts"],
    "pkt_count_bwd":     ["Total Bwd packets",             "Total Backward Packets",
                          "Tot Bwd Pkts"],
    "byte_count_fwd":    ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "byte_count_bwd":    ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "pkt_rate":          ["Flow Packets/s",                "Flow Pkts/s"],
    "byte_rate":         ["Flow Bytes/s",                  "Flow Byts/s"],
    "fwd_pkt_rate":      ["Fwd Packets/s",                 "Fwd Pkts/s"],
    "bwd_pkt_rate":      ["Bwd Packets/s",                 "Bwd Pkts/s"],
    "fwd_pkt_len_total": ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "bwd_pkt_len_total": ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "subflow_fwd_pkts":  ["Subflow Fwd Packets",           "Subflow Fwd Pkts"],
    "subflow_bwd_pkts":  ["Subflow Bwd Packets",           "Subflow Bwd Pkts"],
    "pkt_len_min":       ["Packet Length Min",             "Min Packet Length",
                          "Pkt Len Min"],
    "pkt_len_max":       ["Packet Length Max",             "Max Packet Length",
                          "Pkt Len Max"],
    "pkt_len_mean":      ["Packet Length Mean",            "Pkt Len Mean"],
    "pkt_len_std":       ["Packet Length Std",             "Pkt Len Std"],
    "pkt_len_var":       ["Packet Length Variance",        "Pkt Len Var"],
    "fwd_pkt_len_min":   ["Fwd Packet Length Min",         "Fwd Pkt Len Min"],
    "fwd_pkt_len_max":   ["Fwd Packet Length Max",         "Fwd Pkt Len Max"],
    "fwd_pkt_len_mean":  ["Fwd Packet Length Mean",        "Fwd Pkt Len Mean"],
    "fwd_pkt_len_std":   ["Fwd Packet Length Std",         "Fwd Pkt Len Std"],
    "bwd_pkt_len_min":   ["Bwd Packet Length Min",         "Bwd Pkt Len Min"],
    "bwd_pkt_len_max":   ["Bwd Packet Length Max",         "Bwd Pkt Len Max"],
    "bwd_pkt_len_mean":  ["Bwd Packet Length Mean",        "Bwd Pkt Len Mean"],
    "bwd_pkt_len_std":   ["Bwd Packet Length Std",         "Bwd Pkt Len Std"],
    "iat_mean":          ["Flow IAT Mean"],
    "iat_std":           ["Flow IAT Std"],
    "iat_max":           ["Flow IAT Max"],
    "iat_min":           ["Flow IAT Min"],
    "fwd_iat_mean":      ["Fwd IAT Mean"],
    "fwd_iat_std":       ["Fwd IAT Std"],
    "bwd_iat_mean":      ["Bwd IAT Mean"],
    "bwd_iat_std":       ["Bwd IAT Std"],
    "flag_syn":          ["SYN Flag Count",               "SYN Flag Cnt"],
    "flag_ack":          ["ACK Flag Count",               "ACK Flag Cnt"],
    "flag_fin":          ["FIN Flag Count",               "FIN Flag Cnt"],
    "flag_rst":          ["RST Flag Count",               "RST Flag Cnt"],
    "flag_psh":          ["PSH Flag Count",               "PSH Flag Cnt"],
    "flag_urg":          ["URG Flag Count",               "URG Flag Cnt"],
    "fwd_header_len":    ["Fwd Header Length",            "Fwd Header Len"],
    "init_win_fwd":      ["FWD Init Win Bytes",           "Init_Win_bytes_forward",
                          "Init Fwd Win Bytes",           "Init Fwd Win Byts"],
}
# Coverage: 42/46 (91%) — verified via live smoke test directly against both uploaded sample CSVs
# (cicids-2017-friday.csv, cic-bccc-nrc-tabulariotAttack.csv).
# 4 columns present in the CICIDS-2017 sample only (ICMP Code, ICMP Type,
# Total TCP Flow Time, Attempted Category) and 1 present in the BCCC sample
# only (Attack Name) are NOT canonical features and are intentionally unmapped.
# Unmapped canonical features (8): fwd_byte_rate, bwd_byte_rate (no per-direction
# byte-rate column in either dataset), pkt_count_total / byte_count_total
# (derived in combine() from fwd+bwd, not read from a column).

# ── ALIAS_IOMT ───────────────────────────────────────────────────────────
# CICIoMT-2024 (Train-Test split, D. Kilichev re-upload). 46 columns.
# Verified against: ciciomt-train-set.csv, ciciomt-test-set.csv
# SAME 45-feature schema as the a known-broken re-upload variant, but THIS
# variant carries a genuine, populated "Label" column:
#   train sample Label values: "DoS-TCP"
#   test  sample Label values: "Malformed_Data"
# (Dadkhah et al. 2024 — 19 attack-class string labels + benign; benign
# rows are NOT present in either 10-row sample, so the exact benign string
# is unconfirmed — load_ciciomt() below matches ANY label containing
# "benign" case-insensitively, not just an exact string, to stay robust
# to "Benign" / "BenignTraffic" / "Benign Traffic" variants.)
# Coverage: 17/46 (37%) features ALIAS-RESOLVE correctly after the Srate/Drate
# correction below. The EFFECTIVE coverage, measured empirically as "has at
# least one non-zero value across the full loaded dataset" (the stricter,
# more defensible metric — see Cell 6's feature_coverage_report.csv), is
# 16/46 (35%): "Drate" (-> bwd_pkt_rate) resolves to a real column, but that
# column's values are ~0 across virtually every row in this dataset — which
# independently matches the original CICIoT2023 paper's own published stats
# (Drate mean=5.46e-6, median/25/50/75th percentiles all exactly 0). This is
# a genuine property of the data, not a mapping defect. Separately, "Srate"
# (-> fwd_pkt_rate) and "Rate" (-> pkt_rate) were found to be numerically
# IDENTICAL in this dataset (again matching the source paper's stats table
# exactly) — both alias-resolve and both carry signal, but they are
# redundant with each other, not independent features. Report 16/46 (35%)
# as the dataset's effective coverage in the paper, not 17/46 — it is the
# number that will reproduce if a reviewer re-runs this pipeline.
# Protocol-presence flags (HTTP, HTTPS, TCP, UDP etc.) and statistical
# moment features (Magnitue, Radius, Covariance, Weight) have no
# CICFlowMeter equivalents — correctly zero-filled.
ALIAS_IOMT = {
    "pkt_count_total":  ["Number", "number"],
    "byte_count_total": ["Tot sum",  "tot sum",  "Tot size", "tot size"],
    "pkt_rate":         ["Rate",     "rate"],
    "fwd_pkt_rate":     ["Srate",    "srate"],   # see note below
    "bwd_pkt_rate":     ["Drate",    "drate"],   # see note below
    "pkt_len_min":      ["Min",      "min"],
    "pkt_len_max":      ["Max",      "max"],
    "pkt_len_mean":     ["AVG",      "avg"],
    "pkt_len_std":      ["Std",      "std"],
    "pkt_len_var":      ["Variance", "variance"],
    "iat_mean":         ["IAT",      "iat"],
    "flag_syn":         ["syn_flag_number", "syn_count"],
    "flag_ack":         ["ack_flag_number", "ack_count"],
    "flag_fin":         ["fin_flag_number", "fin_count"],
    "flag_rst":         ["rst_flag_number", "rst_count"],
    "flag_psh":         ["psh_flag_number"],
    "fwd_header_len":   ["Header_Length",   "header_length"],
}
# Srate / Drate added after cross-checking against the CICIoT2023 extractor's
# own published feature definitions (Neto et al. 2023, Sensors 23(13):5941;
# corroborated by multiple independent secondary sources): "Srate = rate of
# outbound packets' transmission in a flow", "Drate = rate of inbound
# packets' transmission in a flow" — directly equivalent to fwd_pkt_rate /
# bwd_pkt_rate (forward/backward packet rate), just computed over this
# extractor's sliding-window unit instead of a CICFlowMeter 5-tuple flow.
#
# Explicitly REJECTED: mapping the "Duration" column to "flow_duration".
# It looks tempting from the name alone, but the same paper's feature table
# documents "Duration" as the packet's IP Time-To-Live (TTL) — confirmed by
# its value range (0-255, matching an 8-bit TTL field) in the paper's
# descriptive statistics table. Mapping it to flow_duration would be a
# genuine semantic error (different physical quantity), not a coverage gain.
# This dataset's actual "flow_duration" column (present in the official
# 47-feature CICIoT2023 schema) is simply ABSENT from this 46-col CICIoMT2024
# re-upload — a genuine schema gap, not an alias-mapping oversight.
#
# Also explicitly NOT mapped (genuine schema mismatch, verified against the
# same source): Protocol Type (categorical, not a canonical feature),
# ece_flag_number / cwr_flag_number (no ECE/CWR slots in the 46-feature
# vocabulary), all protocol-presence one-hot flags (HTTP/HTTPS/DNS/.../LLC —
# no protocol-indicator slot exists in the vocabulary), and Magnitude /
# Radius / Covariance / Weight (composite cross-direction statistics with
# no single-feature canonical equivalent — e.g. Weight is explicitly defined
# as incoming_packets x outgoing_packets, a multiplicative composite that
# does not correspond to any additive canonical count we have).
# Coverage after this correction: 17/46 (37%) — up from 15/46, verified by
# re-running build_semantic_vector() against the real sample CSVs.
# NOTE: Short aliases (Min, Max, AVG, Std, IAT, Rate) are length < 5 chars.
# They are correctly handled by the exact case-insensitive match in
# build_semantic_vector() step 2, which runs BEFORE the substring guard.
# The substring guard in step 3 (len < 5 → skip) only applies to substring
# fallback — exact alias match is always attempted first and has no length guard.
# Coverage: 15/46 (genuine ceiling — extractor design limitation)

# ── ALIAS_UNSW ────────────────────────────────────────────────────────────
# UNSW-NB15 (Moustafa & Slay, MilCIS 2015). Argus + Bro/Zeek extractor — a
# THIRD, genuinely distinct extractor family (neither CICFlowMeter nor the
# CIC-IoT sliding-window statistical extractor). This is the heterogeneity
# upgrade: BRIDGE now spans three independent flow-extraction toolchains.
# Enterprise/general network testbed (NOT IoT-specific — same scope caveat as
# CICIDS-2017; see UNSW_SCOPE_NOTE below).
#
# Verified against the CLEAN labelled split shipped by the Kaggle mirror
# (mrwellsdavid/unsw-nb15): UNSW_NB15_training-set.csv / _testing-set.csv —
# 45 columns WITH a header, binary numeric `label` (0=normal, 1=attack) and a
# human-readable `attack_cat` family string. load_unsw_nb15() reads ONLY files
# that expose BOTH a `label` and a `dur` header column, which automatically
# rejects the raw headerless UNSW-NB15_1..4.csv captures and the
# NUSW-NB15_features.csv metadata file.
#
# Mappings follow the ORIGINAL 2015 feature definitions (Moustafa & Slay),
# mapped by PHYSICAL quantity, not by name:
#   dur    = record total duration                    -> flow_duration
#   spkts  = source->dest packet count                -> pkt_count_fwd
#   dpkts  = dest->source packet count                -> pkt_count_bwd
#   sbytes = source->dest transaction bytes           -> byte_count_fwd
#                                                        + fwd_pkt_len_total
#   dbytes = dest->source transaction bytes           -> byte_count_bwd
#                                                        + bwd_pkt_len_total
#   smean  = mean packet size, source direction       -> fwd_pkt_len_mean
#   dmean  = mean packet size, dest direction         -> bwd_pkt_len_mean
#   sinpkt = source inter-packet arrival time (mSec)  -> fwd_iat_mean
#   dinpkt = dest inter-packet arrival time (mSec)    -> bwd_iat_mean
#   sload  = source data rate (BITS/sec)              -> fwd_byte_rate  [A]
#   dload  = dest data rate (BITS/sec)                -> bwd_byte_rate  [A]
#   swin   = source TCP window advertisement value    -> init_win_fwd   [B]
#
# [A] sload/dload are BITS/sec; fwd_byte_rate/bwd_byte_rate are BYTES/sec — a
#     constant 8x unit factor fully absorbed by the per-feature RobustScaler
#     (scale-invariant), so the mapping is honest at the level of
#     "forward/backward throughput". The unit difference is noted, not hidden.
# [B] swin is the advertised TCP receive-window (source side); init_win_fwd
#     (CICFlowMeter) is the initial forward-window byte count. Both encode the
#     forward TCP window size; mapped with this caveat noted.
#
# EXPLICITLY REJECTED (kept unmapped on purpose — Section 3.4.4 discipline):
#   * rate -> pkt_rate : the derived `rate` column's exact physical definition
#     is NOT in Moustafa & Slay's canonical 49-feature table (added only in the
#     derived train/test split). Rather than guess it is the flow packet-rate,
#     it is left unmapped. (Loss: 1 slot.)
#   * sjit/djit -> iat_std : jitter (mean deviation of packet delay) and the
#     std of inter-arrival times are correlated but NOT the same statistic. No
#     canonical "jitter" slot exists; mapping to iat_std would be a semantic
#     approximation, so it is rejected. (Loss: 2 slots.)
#   * synack/ackdat -> flag_* : these are TCP setup *times* (SYN->SYN_ACK,
#     SYN_ACK->ACK), NOT flag counts. UNSW-NB15's clean split carries NO TCP
#     flag-count columns (only a categorical `state` and these setup times), so
#     ALL flag_* slots are correctly left unmapped — a genuine schema gap, not
#     an oversight. This differing coverage profile vs the CICFlowMeter datasets
#     is exactly the point of a heterogeneous benchmark.
#   * sttl/dttl (IP TTL), stcpb/dtcpb (TCP base seq), tcprtt, sloss/dloss, dwin,
#     proto/service/state (categorical), ct_* (connection-count aggregates),
#     trans_depth/response_body_len (HTTP), is_* : no canonical flow-statistic
#     equivalent — correctly zero-filled.
#
# Effective coverage: 14/46 canonical slots resolve (12 distinct source
# columns; sbytes/dbytes each fill two slots, exactly as ALIAS_FLOWMETER
# dual-maps byte_count_* and *_pkt_len_total). pkt_count_total is additionally
# derived in combine() from fwd+bwd (not read from a column), same as the other
# datasets. Confirm the live number via Cell 6's coverage table and the Cell 2b
# diagnostic after attaching the real file.
ALIAS_UNSW = {
    "flow_duration":     ["dur"],
    "pkt_count_fwd":     ["spkts"],
    "pkt_count_bwd":     ["dpkts"],
    "byte_count_fwd":    ["sbytes"],
    "byte_count_bwd":    ["dbytes"],
    "fwd_pkt_len_total": ["sbytes"],
    "bwd_pkt_len_total": ["dbytes"],
    "fwd_pkt_len_mean":  ["smean"],
    "bwd_pkt_len_mean":  ["dmean"],
    "fwd_iat_mean":      ["sinpkt"],
    "bwd_iat_mean":      ["dinpkt"],
    "fwd_byte_rate":     ["sload"],
    "bwd_byte_rate":     ["dload"],
    "init_win_fwd":      ["swin"],
}

ALIAS_MAPS = {
    "CICIDS-2017":                          ALIAS_FLOWMETER,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  ALIAS_FLOWMETER,
    "CICIoMT-2024":                         ALIAS_IOMT,
    "UNSW-NB15":                             ALIAS_UNSW,
}

def build_semantic_vector(df, ds_name):
    alias_map = ALIAS_MAPS.get(ds_name, {})
    col_lower = {c.strip().lower(): c for c in df.columns}
    out = np.zeros((len(df), N_SEM), dtype=np.float32)
    matched = []
    for sem_name in SEMANTIC_FEATURES:
        si = SEM_IDX[sem_name]
        col = None
        # 1. Exact match (case-insensitive)
        if sem_name in col_lower:
            col = col_lower[sem_name]
        # 2. Alias match (exact, case-insensitive — no length guard here)
        elif sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if al in col_lower:
                    col = col_lower[al]; break
        # 3. Substring fallback — MINIMUM LENGTH GUARD (len < 5 → skip)
        # Short aliases like "Min", "Max", "Std", "IAT", "Rate" must NOT
        # match via substring — prevents false hits on unrelated columns.
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if len(al) < 5:
                    continue
                ms = [c for c in df.columns if al in c.strip().lower()]
                if ms: col = ms[0]; break
        # 3b. Normalised substring fallback (strip spaces/underscores)
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower().replace(' ','').replace('_','')
                if len(al) < 5:
                    continue
                for c in df.columns:
                    cn = c.strip().lower().replace(' ','').replace('_','')
                    if al == cn or (len(al) > 4 and al in cn):
                        col = c; break
                if col: break
        if col is not None:
            vals = pd.to_numeric(df[col], errors='coerce').fillna(0).values
            out[:, si] = vals.astype(np.float32)
            matched.append(sem_name)
    return out, matched

print(f"Alias maps defined for {len(ALIAS_MAPS)} datasets")
for ds, am in ALIAS_MAPS.items():
    print(f"  {ds}: {len(am)} alias entries / {N_SEM} canonical = "
          f"{len(am)/N_SEM*100:.0f}% (alias entries; actual coverage verified at runtime)")
print()
print("NOTE: actual coverage verified at runtime by build_semantic_vector().")
print("Run Cell 2b (diagnostic) after Cell 6 to confirm per-dataset coverage.")
print("Short aliases (<5 chars) only match via exact alias lookup, NOT substring.")

# ── ZERO-FILL SEMANTICS NOTE (paper Section 3.4.1) ───────────────────────────
# Absent features are zero-filled: conflates "true value is zero" with
# "feature not observed". This is a known limitation, disclosed in the paper.
# Mitigation: combine()._balance() enforces 1:1 class balance within each
# dataset block, removing the shortcut-learning risk. Cell 7b verifies this
# empirically. Post-scaling zero-fill displacement is bounded by clip(±10).
# CICIDS-2017 scope note: CICIDS-2017 is a general enterprise IDS benchmark
# (Sharafaldin et al., ICISSP 2018), not an IoT-specific botnet dataset.
# It is retained for canonical vocabulary coverage (91%) and cross-extractor
# domain shift testing. See paper Section 3.3.1 and Section 6 (Limitations).
ZERO_FILL_NOTE = (
    "Absent features zero-filled — conflation with true-zero is a known "
    "limitation. See Cell 7b shortcut test and paper Section 3.4.1."
)
UNSW_SCOPE_NOTE = (
    "UNSW-NB15 is a general enterprise/network testbed (Moustafa & Slay 2015), "
    "NOT IoT-specific. Retained as a THIRD extractor family (Argus/Bro) for "
    "cross-extractor generalisation — same scope caveat as CICIDS-2017."
)
CICIDS2017_SCOPE_NOTE = (
    "CICIDS-2017 is a general enterprise IDS benchmark, NOT IoT-specific. "
    "Retained as vocabulary anchor and cross-extractor shift test only."
)
print(f"Zero-fill policy: {ZERO_FILL_NOTE}")
print(f"CICIDS-2017 scope: {CICIDS2017_SCOPE_NOTE}")

Canonical vocabulary: 46 features
Alias maps defined for 4 datasets
  CICIDS-2017: 42 alias entries / 46 canonical = 91% (alias entries; actual coverage verified at runtime)
  CIC-BCCC-NRC-TabularIoTAttacks-2024: 42 alias entries / 46 canonical = 91% (alias entries; actual coverage verified at runtime)
  CICIoMT-2024: 17 alias entries / 46 canonical = 37% (alias entries; actual coverage verified at runtime)
  UNSW-NB15: 14 alias entries / 46 canonical = 30% (alias entries; actual coverage verified at runtime)

NOTE: actual coverage verified at runtime by build_semantic_vector().
Run Cell 2b (diagnostic) after Cell 6 to confirm per-dataset coverage.
Short aliases (<5 chars) only match via exact alias lookup, NOT substring.
Zero-fill policy: Absent features zero-filled — conflation with true-zero is a known limitation. See Cell 7b shortcut test and paper Section 3.4.1.
CICIDS-2017 scope: CICIDS-2017 is a general enterprise IDS benchmark, NOT IoT-specific. Retained as vocabulary anchor an

In [3]:
class Config:
    # ── Kaggle dataset paths ──────────────────────────────────────────────
    # PRIMARY (flow-level, all CICFlowMeter-compatible)
    CICIDS_PATH  = "/kaggle/input/datasets/bertvankeulen/cicids-2017"
    BCCC_PATH    = "/kaggle/input/datasets/kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024"
    IOMT_PATH    = "/kaggle/input/datasets/sinfdosh1990/ciciomt2024"
    UNSW_PATH    = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"

    # Auto-detect
    USE_CICIDS  = os.path.isdir(CICIDS_PATH) if CICIDS_PATH else False
    USE_BCCC    = os.path.isdir(BCCC_PATH)   if BCCC_PATH   else False
    USE_IOMT    = os.path.isdir(IOMT_PATH)   if IOMT_PATH   else False
    USE_UNSW    = os.path.isdir(UNSW_PATH)   if UNSW_PATH   else False

    # ── Row caps per dataset ──────────────────────────────────────────────
    CICIDS_MAX   = 500_000
    BCCC_MAX     = 3_000_000   # BCCC aggregates 9 IoT datasets, 1M+ records
    IOMT_MAX     = 3_000_000
    UNSW_MAX     = 500_000     # UNSW-NB15 train+test ~257k rows (cap not hit)

    # ── Sequence parameters ───────────────────────────────────────────────
    WINDOW_SIZE = 32
    STRIDE      = 4
    MAX_TRAIN_SEQ = 800_000
    MAX_TEST_SEQ  = 200_000
    MAX_VAL_SEQ   = 200_000

    # ── Class balance ─────────────────────────────────────────────────────
    TARGET_ATK_BEN_RATIO = 1.0   # strict 1:1 → ~43% attack windows after windowing

    # ── Training ──────────────────────────────────────────────────────────
    EPOCHS        = 30
    WARMUP        = 3
    EARLY_STOP    = 7
    BATCH_SIZE    = 512
    LR            = 5e-4
    WD            = 1e-4
    FOCAL_GAMMA   = 2.5
    LABEL_SMOOTH  = 0.01
    AUX_WT        = 0.05
    DEEP_SUP_WT   = 0.3     # per-expert CE weight (deep supervision; prevents gate collapse)
    CORAL_WT      = 0.5     # source-domain CORAL alignment weight (0 disables; ablatable)
    TTBN_ADAPT    = True    # transductive test-time BatchNorm adaptation at LODO eval

    # ── Architecture ──────────────────────────────────────────────────────
    EMBED_DIM     = 32
    CONV_CH       = [64, 128, 128]
    GRU_HIDDEN    = 128
    GRU_LAYERS    = 2
    ATTN_HEADS    = 8
    DROPOUT       = 0.20
    CBGAF_DIM     = 128

    # ── Evaluation ────────────────────────────────────────────────────────
    EVAL_SEEDS     = [42, 123, 456, 789, 2024]
    FAST_SEEDS     = [42, 123, 456]
    # REVISION (R1-M1): baselines use the SAME 5 seeds as TCH-Net so every
    # in-distribution paired test is matched seed-for-seed (n = 5 pairs).
    BASE_SEEDS     = [42, 123, 456, 789, 2024]
    LODO_SEEDS     = [42, 123, 456]   # 3 seeds per fold (descriptive spread only; no paired test)
    ABL_EPOCHS     = 30
    BASE_EPOCHS    = 30
    LODO_EPOCHS    = 30
    WARMUP_FAST    = 5

    N_DEV_CATS = 6   # 0=sensor, 1=camera, 2=appliance, 3=IIoT, 4=server, 5=unknown

    # ── Realistic deployment prior ─────────────────────────────────
    # Attack prevalence in real IoT traffic is typically 1-10%.
    # We evaluate at 5% to simulate realistic deployment conditions.
    REALISTIC_ATK_PRIOR = 0.05   # P(attack) in realistic traffic

    # ── HP sensitivity ranges ──────────────────
    HP_SENS_CONFIGS = [
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.20},   # default
        {'lr': 1e-3, 'wd': 1e-4, 'dropout': 0.20},   # 2x LR
        {'lr': 2e-4, 'wd': 1e-4, 'dropout': 0.20},   # 0.4x LR
        {'lr': 5e-4, 'wd': 1e-3, 'dropout': 0.20},   # 10x WD
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.30},   # higher dropout
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.10},   # lower dropout
    ]
    HP_SENS_SEEDS = [42, 123]   # 2 seeds × 6 configs = 12 runs
    N_CLASSES  = 2   # benign vs attack (binary IDS)
    N_DS_SRC   = 4   # CICIDS-2017=0, BCCC=1, CICIoMT-2024=2, UNSW-NB15=3

    # Device category map: dataset → {device_id: category_id}
    # device_id is an arbitrary per-loader integer; category is one of N_DEV_CATS
    DEVICE_CAT_MAP = {
        "CICIDS-2017":                          {0: 4},   # 0→server/workstation
        "CIC-BCCC-NRC-TabularIoTAttacks-2024":  {1: 0},   # 1→IoT sensor/device (mixed: cameras, lights, sensors)
        "CICIoMT-2024":                         {2: 3},   # 2→IIoT/medical device
        "UNSW-NB15":                            {3: 4},   # 3→server/workstation (enterprise, non-IoT)
    }

    # ── Reviewer-requested fixes ──────────────────────────────────────────────
    # chronological split is the DEFAULT.
    SPLIT_MODE      = 'chronological'   # 'chronological' | 'random'

    # REVISION (R1-m1): explicit validation subset for early stopping and
    # checkpoint selection. Carved chronologically per dataset BEFORE windowing:
    # train | val | test = 70 | 10 | 20 % of rows in row order. The test rows are
    # identical to the previous two-way split and are never used for selection.
    VAL_FRAC        = 0.10
    # LODO: validation is carved the same way from the SOURCE datasets only —
    # the held-out target domain is never used for early stopping.
    LODO_VAL_FRAC   = 0.10

    # feature-availability mask
    USE_MISS_MASK   = True

    # common seed set for all paired tests (REVISION R1-M1: 5 matched seeds)
    COMMON_SEEDS    = [42, 123, 456, 789, 2024]
    STAT_ALPHA      = 0.05   # family-wise level for Bonferroni-corrected paired tests

    # IDS operational metrics
    FIXED_FPR_LEVELS   = [0.001, 0.01, 0.05]
    FIXED_TPR_LEVELS   = [0.95, 0.99]  # also computes FPR@TPR99 for every model

    OUT = "tch_net_v2_results"

    # ── LODO baseline names (5 DL models run in LODO alongside TCH-Net) ────
    # Transformer-IDS, BiGRU-IDS, MLP-IDS, CNN-LSTM, 1D-CNN-IDS
    LODO_BASELINE_NAMES = [
        "Transformer-IDS",
        "BiGRU-IDS",
        "MLP-IDS",
        "CNN-LSTM",
        "1D-CNN-IDS",
    ]

os.makedirs(Config.OUT, exist_ok=True)
PAL = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
       '#00BCD4','#795548','#E91E63','#607D8B','#CDDC39']
def SAVE(n, fig=None):
    """Save current figure and display inline in Jupyter."""
    _f = fig if fig is not None else plt.gcf()
    _f.savefig(os.path.join(Config.OUT, n), bbox_inches='tight', dpi=150)

print("Dataset availability:")
for n, f in [("CICIDS-2017", Config.USE_CICIDS),
             ("CIC-BCCC-NRC-TabularIoTAttacks-2024", Config.USE_BCCC),
             ("CICIoMT-2024", Config.USE_IOMT),
             ("UNSW-NB15", Config.USE_UNSW)]:
    print(f"  {n:<38}: {'FOUND' if f else 'NOT FOUND'} [PRIMARY]")


Dataset availability:
  CICIDS-2017                           : FOUND [PRIMARY]
  CIC-BCCC-NRC-TabularIoTAttacks-2024   : FOUND [PRIMARY]
  CICIoMT-2024                          : FOUND [PRIMARY]
  UNSW-NB15                             : FOUND [PRIMARY]


In [4]:
ATTACK_TYPE_STORE = {}

def _subsample(X, y, max_n, extras=None, seed=42):
    """
    Subsample to at most max_n rows, preserving the original class ratio
    but enforcing a minimum of 5000 samples per class when possible.
    The actual class re-balancing to 1:1 happens later in combine()._balance().
    This function only caps extremely large datasets to avoid OOM.
    """
    if max_n is None or len(X) <= max_n:
        return (X, y) + (tuple(extras) if extras else ())
    rng = np.random.RandomState(seed)
    ben = np.where(y==0)[0]; atk = np.where(y==1)[0]
    r = len(ben) / len(y)
    nb = min(len(ben), max(5_000, int(max_n * r)))
    na = min(len(atk), max(5_000, max_n - nb))
    if nb + na > max_n:
        scale = max_n / (nb + na)
        nb = max(5_000, int(nb * scale))
        na = max(5_000, min(max_n - nb, int(na * scale)))
    nb = min(nb, len(ben)); na = min(na, len(atk))
    keep = np.concatenate([rng.choice(ben, nb, replace=False),
                           rng.choice(atk, na, replace=False)])
    rng.shuffle(keep)
    result = [X[keep], y[keep]]
    if extras:
        for a in extras: result.append(a[keep])
    print(f"  Subsampled {len(X):,} -> {len(keep):,} "
          f"(ben={nb:,} atk={na:,} atk%={na/(nb+na)*100:.1f}%)")
    return tuple(result)

def augment_minority(X, y, target=5000, noise_std=0.01, seed=42):
    rng = np.random.RandomState(seed)
    ben_idx = np.where(y == 0)[0]
    atk_idx = np.where(y == 1)[0]
    minority_idx = ben_idx if len(ben_idx) < len(atk_idx) else atk_idx
    minority_label = 0 if len(ben_idx) < len(atk_idx) else 1
    n_minority = len(minority_idx)
    if n_minority >= target:
        return X, y
    n_needed = target - n_minority
    print(f"    Augmenting minority ({n_minority} -> {target}, "
          f"generating {n_needed} synthetic rows via Gaussian noise sigma={noise_std})")
    chosen = rng.choice(minority_idx, n_needed, replace=True)
    X_aug = X[chosen] + rng.normal(0, noise_std,
                                    size=(n_needed, X.shape[1])).astype(np.float32)
    X_aug = np.clip(X_aug, X.min(axis=0), X.max(axis=0))
    y_aug = np.full(n_needed, minority_label, dtype=np.int32)
    X_new = np.vstack([X, X_aug])
    y_new = np.hstack([y, y_aug])
    print(f"    After augmentation: ben={(y_new==0).sum():,} atk={(y_new==1).sum():,}")
    return X_new, y_new

def _is_benign_mask(lbl_series):
    """
    Robust benign/attack split for an object-dtype label column.
    Matches an exact small set of known benign tokens AND any label that
    merely CONTAINS "benign" (case-insensitive) — covers "Benign",
    "BENIGN", "BenignTraffic", "Benign Traffic", "Benign_Traffic", etc.
    without having to enumerate every uploader's exact string.
    """
    s = lbl_series.astype(str).str.strip().str.lower()
    exact = s.isin(['benign', 'benigntraffic', 'normal', '0'])
    contains = s.str.contains('benign', regex=False)
    return exact | contains

# ── CICIDS-2017 ──────────────────────────────────────────────────────────
# Verified against: cicids-2017-friday.csv, cicids-2017-friday-plus.csv
# (89 / 105 columns — "-plus" adds extra Flow ID / "Local_*" metadata columns
# that are not part of the canonical feature set and are safely ignored).
# Label column: "Label" (string, e.g. "BENIGN"). Uses ALIAS_FLOWMETER.
def load_cicids2017(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIDS-2017\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True) +
                   glob.glob(os.path.join(data_dir,"**","*.parquet"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    for fp in files:
        print(f"  Loading {os.path.basename(fp)}...")
        try:
            df = pd.read_parquet(fp) if fp.endswith('.parquet') else pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None: print(f"    No label column"); continue
            y = (~_is_benign_mask(df[lc])).values.astype(np.int32)
            atk = np.where(y==0, 'BENIGN', df[lc].astype(str).str.strip().values)
            X_sem, matched = build_semantic_vector(df, "CICIDS-2017")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            print(f"    {len(y):,} rows | coverage={len(matched)}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
        except Exception as e: print(f"    Skip: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 0 → server/workstation category (Config.DEVICE_CAT_MAP["CICIDS-2017"])
    dev = np.zeros(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CICIDS-2017'] = atk
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# ── CIC-BCCC-NRC-TabularIoTAttacks-2024 ──────────────────────────────────
# Replaces the broken CIC-IoT-DIAD-2024 (Mostafa Ali Ogab) — that dataset's
# Label column was "NeedManualLabel" on EVERY row, including attack-named
# files. Confirmed unusable; see Cell 2 dataset-reset note.
#
# BCCC research center (York University). CICFlowMeter-extracted, 85
# columns, aggregates 9 IoT datasets into one unified table. 1M+ records.
# Verified against: cic-bccc-nrc-tabulariotAttack.csv
#   Label column: "Label" — int, 0 in the (benign-only) sample.
#   "Attack Name" column: string, e.g. "Benign Traffic" in the sample.
# Label strategy: prefer the explicit "Attack Name" string (unambiguous
# human-readable family) when present; fall back to numeric Label
# (0=benign / nonzero=attack — works whether Label is strictly binary or a
# multi-class integer encoding, since != 0 is correct in both cases).
# Kaggle: kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024
def load_cic_bccc_nrc(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CIC-BCCC-NRC-TabularIoTAttacks-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    _agree_n = 0; _agree_match = 0   # label cross-validation tally (see below)
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True

            attack_col = next((c for c in df.columns if c.lower().strip() == 'attack name'), None)
            label_col  = next((c for c in df.columns if c.lower().strip() == 'label'), None)

            # ── Label cross-validation ──────────────────────────────────────
            # This dataset carries TWO independent label signals per row:
            # a human-readable family string ("Attack Name") and a numeric
            # code ("Label"). Rather than trusting one and discarding the
            # other, derive y from BOTH whenever both are present and report
            # their agreement rate. Attack Name is used as the actual ground
            # truth (explicit, unambiguous string) — Label is a corroborating
            # signal, not a silent fallback. Any disagreement is surfaced
            # immediately instead of being silently absorbed into the labels.
            if attack_col is not None and label_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y_from_attack = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                lbl = df[label_col]
                if lbl.dtype == object:
                    y_from_label = (~_is_benign_mask(lbl)).values.astype(np.int32)
                else:
                    y_from_label = (lbl.values != 0).astype(np.int32)
                _agree = (y_from_attack == y_from_label)
                _agree_n += len(_agree); _agree_match += int(_agree.sum())
                if not _agree.all():
                    print(f"    NOTE: {int((~_agree).sum())}/{len(_agree)} rows in "
                          f"{os.path.basename(fp)} disagree between 'Attack Name' "
                          f"and numeric 'Label' — using 'Attack Name'")
                y = y_from_attack
                atk = atk_str.values
            elif attack_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                atk = atk_str.values
            elif label_col is not None:
                lbl = df[label_col]
                if lbl.dtype == object:
                    y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                    atk = lbl.astype(str).values
                else:
                    y = (lbl.values != 0).astype(np.int32)
                    atk = np.where(y==0, 'Benign Traffic', lbl.astype(str).values)
            else:
                print(f"  No 'Attack Name' or 'Label' column in {os.path.basename(fp)} — skip")
                continue

            X_sem, matched = build_semantic_vector(df, "CIC-BCCC-NRC-TabularIoTAttacks-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    if _agree_n > 0:
        _agree_pct = _agree_match / _agree_n * 100
        print(f"\n  Label cross-validation ('Attack Name' vs numeric 'Label'): "
              f"{_agree_match:,}/{_agree_n:,} rows agree ({_agree_pct:.2f}%)")
        if _agree_pct < 99.0:
            print(f"  WARNING: >1% disagreement between the two label columns — "
                  f"inspect before trusting either derivation in the paper.")
    # device_id 1 → generic IoT device category (BCCC mixes cameras/lights/sensors)
    dev = np.ones(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CIC-BCCC-NRC-TabularIoTAttacks-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    rng_bccc = np.random.RandomState(42)
    perm = rng_bccc.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# ── CICIoMT-2024 (Train-Test, D. Kilichev) ───────────────────────────────
# Replaces the broken CICIoMT-2024 (Lian Mollick) variant — that upload had
# 45 columns and NO label column at all (same reduced schema as CIC-IoT-2023).
# Confirmed unusable; see Cell 2 dataset-reset note.
#
# Lashkari et al. (2024). Custom CIC statistical extractor (46-col schema
# incl. Label — identical feature schema to the broken variant, but THIS
# upload's Label column is genuinely populated).
# Verified against: ciciomt-train-set.csv ("DoS-TCP"), ciciomt-test-set.csv
# ("Malformed_Data"). Neither 10-row sample contains a benign row, so the
# exact benign string token is unconfirmed — _is_benign_mask() matches any
# label containing "benign" case-insensitively to stay robust to whichever
# exact spelling ("Benign" / "BenignTraffic" / "Benign Traffic") this
# upload uses.
# Label column: "Label". 17/46 features alias-resolve; 16/46 (35%) carry
# non-trivial signal on the full live dataset (Drate/bwd_pkt_rate resolves
# correctly but is ~0 across virtually every real row — matches the source
# paper's own published stats, not a mapping defect). See Cell 2 for the
# full Srate/Drate/Duration reasoning.
def load_ciciomt(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIoMT-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None:
                # Some file splits carry no label — skip silently
                continue
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                atk = lbl.astype(str).values
            else:
                y = (lbl.values != 0).astype(np.int32)
                atk = np.where(y==0, 'Benign', lbl.astype(str).values)
            X_sem, matched = build_semantic_vector(df, "CICIoMT-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 2 → IIoT/medical device category
    dev = np.full(len(y), 2, dtype=np.int32)
    ATTACK_TYPE_STORE['CICIoMT-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    rng_iomt = np.random.RandomState(42)
    perm = rng_iomt.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# \u2500\u2500 UNSW-NB15 (Moustafa & Slay 2015; Argus + Bro/Zeek) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# THIRD extractor family \u2014 the heterogeneity upgrade. Enterprise/general
# network testbed (NOT IoT-specific \u2014 same scope caveat as CICIDS-2017).
# Reads ONLY the clean labelled split (files exposing BOTH a `label` and a
# `dur` header column): UNSW_NB15_training-set.csv / _testing-set.csv.
# This automatically SKIPS:
#   * the raw headerless captures UNSW-NB15_1..4.csv (no header -> first data
#     row misread as header -> lacks `label`/`dur` -> rejected), and
#   * NUSW-NB15_features.csv / *LIST_EVENTS* metadata files.
# Label: numeric `label` (0=normal, 1=attack). `attack_cat` gives the family.
# Uses ALIAS_UNSW. device_id 3 -> enterprise/server category (DEVICE_CAT_MAP).
# Kaggle: mrwellsdavid/unsw-nb15
def load_unsw_nb15(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING UNSW-NB15\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description','list_events',
                        'list-events','ground_truth','_gt'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            cols_lower = {c.lower() for c in df.columns}
            # Guard: accept ONLY the clean labelled split. Raw headerless
            # captures and the features-metadata file lack this exact pair.
            if 'label' not in cols_lower or 'dur' not in cols_lower:
                print(f"  Skip {os.path.basename(fp)}: not the clean labelled "
                      f"schema (needs 'label'+'dur' header) \u2014 raw/meta file")
                continue
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next(c for c in df.columns if c.lower().strip() == 'label')
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
            else:
                y = (lbl.values != 0).astype(np.int32)   # 0=normal, 1=attack
            ac = next((c for c in df.columns if c.lower().strip() == 'attack_cat'), None)
            if ac is not None:
                atk = np.where(y==0, 'Normal', df[ac].astype(str).str.strip().values)
            else:
                atk = np.where(y==0, 'Normal', 'Attack')
            X_sem, matched = build_semantic_vector(df, "UNSW-NB15")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
        except Exception as e:
            print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 3 -> enterprise/server category (non-IoT, like CICIDS-2017)
    dev = np.full(len(y), 3, dtype=np.int32)
    ATTACK_TYPE_STORE['UNSW-NB15'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5 or atk_pct > 95:
        print(f"  WARNING: extreme attack% ({atk_pct:.1f}%) \u2014 check label parsing")
    rng_unsw = np.random.RandomState(42)
    perm = rng_unsw.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

print("All 4 dataset loaders defined")


All 4 dataset loaders defined


In [5]:
class MultiDatasetLoader:
    # Warn if combined post-balance count < 2*MIN_FLOOR (signals scarce dataset).
    # Any dataset whose minority class is small will be flagged but still included.
    MIN_FLOOR = 5_000  # advisory threshold only

    def __init__(self):
        self.datasets = {}
        self.scaler = None
        # NOTE: X_train_scaled / X_test_scaled (stored after scaling)
        # Named *_raw to match usage in HP sensitivity cells.
        self.X_train_raw = self.X_test_raw = self.X_val_raw = None

    def add(self, name, X, y, dev, ds_src_id):
        if X is None: print(f"  Skip {name}: no data"); return
        cat_map = Config.DEVICE_CAT_MAP.get(name, {})
        dev_cats = np.array([cat_map.get(int(d), 5) for d in dev], dtype=np.int32)
        ds_ids   = np.full(len(y), ds_src_id, dtype=np.int32)
        ctx      = np.stack([ds_ids, dev_cats], axis=1)
        n_matched = int((X != 0).any(axis=0).sum())
        self.datasets[name] = {'X': X, 'y': y, 'dev': dev, 'ctx': ctx,
                               'ds_src': ds_src_id, 'n': X.shape[0]}
        print(f"  Added {name}: {X.shape[0]:,}x{X.shape[1]} "
              f"(non-zero dims: {n_matched}/{N_SEM} = {n_matched/N_SEM*100:.0f}%)")

    def _balance(self, X, y, ctx, ratio, rng, name=""):
        """
        Strict class balance with seeded RNG for reproducibility.

        ratio=1.0 → 50% attack rows → p(window=attack) ≈ 43% with majority-vote(w=32).

        MIN_FLOOR: advisory only — warns when a dataset contributes fewer than
        2*MIN_FLOOR total samples after balancing (poor signal, still included).
        The actual cap is ALWAYS strict: majority capped to ratio * minority.
        We never inflate the majority beyond ratio*minority just because minority
        is small — that would produce up to 5:1 imbalance and near-100% attack
        windows after windowing (empirically verified on prior, since-removed
        low-sample datasets in this pipeline's history).
        FocalLoss alpha weighting handles residual class imbalance at batch level.
        """
        ben = np.where(y == 0)[0]
        atk = np.where(y == 1)[0]
        if len(ben) == 0 or len(atk) == 0:
            print(f"  WARNING {name}: single-class data "
                  f"(ben={len(ben)}, atk={len(atk)}) — skipping balance")
            return X, y, ctx

        # Strict 1:1 balance always — even if minority class is tiny.
        # Reason: majority-vote windowing needs ~50% row balance to produce
        # mixed-label windows. A 20:1 imbalance produces ~100% attack windows,
        # giving the model zero benign examples from that dataset.
        # A dataset with very few minority-class rows is accepted as-is rather
        # than upsampled past this floor — FocalLoss alpha handles the residual.
        # FocalLoss alpha and the global _cap handle residual imbalance.
        MIN_SAMPLES = 5000
        if len(atk) > ratio * len(ben):
            if len(ben) >= MIN_SAMPLES:
                n_atk = int(ratio * len(ben))
                atk = rng.choice(atk, n_atk, replace=False)
            else:
                n_atk = min(len(atk), MIN_SAMPLES)
                atk = rng.choice(atk, n_atk, replace=False)
        elif len(ben) > ratio * len(atk):
            if len(atk) >= MIN_SAMPLES:
                n_ben = int(ratio * len(atk))
                ben = rng.choice(ben, n_ben, replace=False)
            else:
                n_ben = min(len(ben), MIN_SAMPLES)
                ben = rng.choice(ben, n_ben, replace=False)

        n_total = len(ben) + len(atk)
        if n_total < 2 * self.MIN_FLOOR:
            print(f"  ADVISORY {name}: only {n_total:,} samples after balancing "
                  f"(minority class was very small: {min(len(ben),len(atk)):,}). "
                  f"FocalLoss alpha weighting compensates during training.")

        keep = np.concatenate([ben, atk])
        # CRITICAL FIX: DO NOT shuffle here.
        # combine() performs a chronological split by sorting positional indices
        # within each dataset block. Shuffling here destroys temporal order
        # within the block, making ds_idx_sorted meaningless as a time-order
        # proxy. Instead, sort keep so rows are in their original temporal order.
        # _cap() shuffles windows after sequencing — randomness is preserved there.
        keep = np.sort(keep)
        return X[keep], y[keep], ctx[keep]

    def combine(self, ratio=None, test_size=0.2, seed=42):
        if ratio is None: ratio = Config.TARGET_ATK_BEN_RATIO
        rng = np.random.RandomState(seed)
        print("\n" + "="*70 + "\nCOMBINE & PREPROCESS\n" + "="*70)
        Xa, ya, ca, da = [], [], [], []

        for di, (name, d) in enumerate(self.datasets.items()):
            X_r = d['X'].copy(); y_r = d['y'].copy(); c_r = d['ctx'].copy()
            np.nan_to_num(X_r, copy=False, nan=0., posinf=1e6, neginf=-1e6)

            n_nonzero = int((X_r != 0).any(axis=0).sum())
            if n_nonzero == 0:
                print(f"  SKIP {name}: 0% feature coverage — all features zero.")
                continue
            if n_nonzero < 5:
                print(f"  WARNING {name}: very low coverage ({n_nonzero}/46). Results unreliable.")

            # ── FIX: derive pkt_count_total BEFORE scaling ───────────────────
            # Cannot do fwd_scaled + bwd_scaled ≠ scale(fwd+bwd). Must use raw counts.
            _fwd = SEM_IDX["pkt_count_fwd"]
            _bwd = SEM_IDX["pkt_count_bwd"]
            _tot = SEM_IDX["pkt_count_total"]
            zero_tot = (X_r[:, _tot] == 0)
            X_r[zero_tot, _tot] = X_r[zero_tot, _fwd] + X_r[zero_tot, _bwd]

            X_r, y_r, c_r = self._balance(X_r, y_r, c_r, ratio, rng, name)
            ben_r = int((y_r==0).sum()); atk_r = int((y_r==1).sum())
            est_seq = max(0, (len(y_r) - Config.WINDOW_SIZE)) // Config.STRIDE + 1
            print(f"  {name}: {len(y_r):,} rows "
                  f"(ben={ben_r:,} atk={atk_r:,} "
                  f"ratio={atk_r/max(ben_r,1):.1f}:1) "
                  f"est_sequences~{est_seq:,}")
            Xa.append(X_r); ya.append(y_r); ca.append(c_r)
            da.append(np.full(len(y_r), di, dtype=np.int32))

        if not Xa:
            raise RuntimeError("No datasets survived combine() — check feature coverage")

        X_c   = np.vstack(Xa).astype(np.float32)
        y_c   = np.hstack(ya).astype(np.int32)
        ctx_c = np.vstack(ca).astype(np.int32)
        ds_c  = np.hstack(da).astype(np.int32)

        total_ben = int((y_c == 0).sum()); total_atk = int((y_c == 1).sum())
        print(f"\n  Pre-split combined: {len(y_c):,} rows "
              f"| ben={total_ben:,} ({total_ben/len(y_c)*100:.1f}%) "
              f"| atk={total_atk:,} ({total_atk/len(y_c)*100:.1f}%)")

        # ── CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT (default) ──────────
        # REVISION (R1-m1). Split BEFORE window generation, per dataset, by
        # original row order:
        #   first (1 − test_size − VAL_FRAC) of rows → train
        #   next   VAL_FRAC                         → validation (early stopping
        #                                             and checkpoint selection ONLY)
        #   last   test_size                        → test (same rows as the
        #                                             previous two-way split)
        # Windows are generated separately inside each partition and never across
        # datasets, so no sliding window can contain rows from two partitions:
        # overlapping-window leakage between train, validation and test is
        # impossible by construction. Previously there was no validation subset
        # and early stopping monitored the test set.
        _vf = getattr(Config, 'VAL_FRAC', 0.10)
        if Config.SPLIT_MODE == 'chronological':
            i_tr, i_va, i_te = [], [], []
            for ds_id in np.unique(ds_c):
                # _balance() keeps rows in original (temporal) order and datasets
                # are concatenated block-wise, so sorted positions = time order.
                ds_idx_sorted = np.sort(np.where(ds_c == ds_id)[0])
                n_trva = int(len(ds_idx_sorted) * (1 - test_size))
                n_va   = int(len(ds_idx_sorted) * _vf)
                i_tr.append(ds_idx_sorted[:n_trva - n_va])
                i_va.append(ds_idx_sorted[n_trva - n_va:n_trva])
                i_te.append(ds_idx_sorted[n_trva:])
            i_tr, i_va, i_te = np.concatenate(i_tr), np.concatenate(i_va), np.concatenate(i_te)
            print(f"  Split mode: CHRONOLOGICAL (per-dataset train/val/test = "
                  f"{(1-test_size-_vf)*100:.0f}/{_vf*100:.0f}/{test_size*100:.0f}% by row order)")
        else:
            # Random split retained for ablation comparisons only
            _idx_all = np.arange(len(y_c))
            _idx_trva, i_te = train_test_split(_idx_all, test_size=test_size,
                                               stratify=y_c, random_state=seed)
            i_tr, i_va = train_test_split(_idx_trva, test_size=_vf / (1 - test_size),
                                          stratify=y_c[_idx_trva], random_state=seed)
            print(f"  Split mode: RANDOM stratified (ablation only — not default)")
        X_tr, y_tr, c_tr, d_tr = X_c[i_tr], y_c[i_tr], ctx_c[i_tr], ds_c[i_tr]
        X_va, y_va, c_va, d_va = X_c[i_va], y_c[i_va], ctx_c[i_va], ds_c[i_va]
        X_te, y_te, c_te, d_te = X_c[i_te], y_c[i_te], ctx_c[i_te], ds_c[i_te]

        # Scale AFTER pkt_count_total derivation (now done above, pre-split).
        # The scaler is fit on TRAIN rows only; validation and test are transformed.
        self.scaler = RobustScaler(quantile_range=(5, 95))
        X_tr = np.clip(self.scaler.fit_transform(X_tr), -10, 10).astype(np.float32)
        X_va = np.clip(self.scaler.transform(X_va),     -10, 10).astype(np.float32)
        X_te = np.clip(self.scaler.transform(X_te),     -10, 10).astype(np.float32)

        # Validate post-split balance
        for tag_, y_ in [('Train', y_tr), ('Val', y_va), ('Test', y_te)]:
            b_ = int((y_==0).sum()); a_ = int((y_==1).sum())
            print(f"  {tag_+':':<6} {len(y_):,} | ben={b_:,} ({b_/len(y_)*100:.1f}%) "
                  f"| atk={a_:,} ({a_/len(y_)*100:.1f}%)")
        # Per-dataset validation composition (disclosed in the paper's split description)
        _names_ = list(self.datasets.keys())
        for ds_id in np.unique(d_va):
            _m_ = (d_va == ds_id)
            _nm_ = _names_[ds_id] if ds_id < len(_names_) else f'DS{ds_id}'
            print(f"    val rows {_nm_:<38}: {int(_m_.sum()):,} "
                  f"(attack {float(y_va[_m_].mean())*100:.1f}%)")

        # Store scaled data for HP sensitivity reuse
        self.X_train_raw = X_tr; self.y_train_raw = y_tr
        self.ctx_train_raw = c_tr; self.ds_train_raw = d_tr
        self.X_val_raw   = X_va; self.y_val_raw   = y_va
        self.ctx_val_raw = c_va; self.ds_val_raw = d_va
        self.X_test_raw  = X_te; self.y_test_raw  = y_te
        self.ctx_test_raw = c_te; self.ds_test_raw = d_te

        print(f"  Scaler: single RobustScaler(q5,q95) fit on train only")
        # Post-split balance check
        for tag_, y_ in [('Train', y_tr), ('Test', y_te)]:
            r_ = float((y_==1).mean())
            if r_ < 0.05 or r_ > 0.95:
                raise RuntimeError(f'{tag_} set is severely imbalanced: attack%={r_*100:.1f}%. Check _balance().')
        _rv = float((y_va==1).mean())
        if _rv < 0.05 or _rv > 0.95:
            print(f"  WARNING: validation set is imbalanced (attack%={_rv*100:.1f}%); "
                  f"early stopping falls back to validation loss if it is single-class.")
        return X_tr, y_tr, c_tr, d_tr, X_te, y_te, c_te, d_te

loader = MultiDatasetLoader()
DS_SRC = {
    "CICIDS-2017":                          0,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  1,
    "CICIoMT-2024":                         2,
    "UNSW-NB15":                             3,
}
print("MultiDatasetLoader ready")
print(f"Dataset source IDs: {DS_SRC}")


MultiDatasetLoader ready
Dataset source IDs: {'CICIDS-2017': 0, 'CIC-BCCC-NRC-TabularIoTAttacks-2024': 1, 'CICIoMT-2024': 2, 'UNSW-NB15': 3}


In [6]:
print("="*70+"\nLOADING DATASETS\n"+"="*70)

if Config.USE_CICIDS:
    try:
        X,y,d = load_cicids2017(Config.CICIDS_PATH, Config.CICIDS_MAX)
        if X is not None: loader.add('CICIDS-2017', X, y, d, DS_SRC['CICIDS-2017'])
        del X,y,d
    except Exception as e: print(f"CICIDS-2017 failed: {e}")

if Config.USE_BCCC:
    try:
        X,y,d = load_cic_bccc_nrc(Config.BCCC_PATH, Config.BCCC_MAX)
        if X is not None: loader.add('CIC-BCCC-NRC-TabularIoTAttacks-2024', X, y, d,
                                      DS_SRC['CIC-BCCC-NRC-TabularIoTAttacks-2024'])
        del X,y,d
    except Exception as e: print(f"CIC-BCCC-NRC-TabularIoTAttacks-2024 failed: {e}")

if Config.USE_IOMT:
    try:
        X,y,d = load_ciciomt(Config.IOMT_PATH, Config.IOMT_MAX)
        if X is not None: loader.add('CICIoMT-2024', X, y, d, DS_SRC['CICIoMT-2024'])
        del X,y,d
    except Exception as e: print(f"CICIoMT-2024 failed: {e}")

if Config.USE_UNSW:
    try:
        X,y,d = load_unsw_nb15(Config.UNSW_PATH, Config.UNSW_MAX)
        if X is not None: loader.add('UNSW-NB15', X, y, d, DS_SRC['UNSW-NB15'])
        del X,y,d
    except Exception as e: print(f"UNSW-NB15 failed: {e}")

gc.collect()
DS_NAMES = list(loader.datasets.keys())
print(f"\n{len(DS_NAMES)} datasets loaded: {DS_NAMES}")
assert len(DS_NAMES) >= 1, "No datasets loaded — check Kaggle input paths in Config"

# ── Feature coverage table (for paper) ───────────────────────────────────
print(f"\n{'='*70}\nFEATURE COVERAGE TABLE (for Section III of paper)\n{'='*70}")
print(f"{'Dataset':<38} {'Matched':>8} {'Coverage':>10} {'Zero-filled features (first 5)'}")
print("-"*70)
for name, data in loader.datasets.items():
    X = data['X']
    covered = int((X != 0).any(axis=0).sum())
    zero_feats = [SEMANTIC_FEATURES[i] for i in range(N_SEM) if (X[:, i] == 0).all()]
    z_str = ', '.join(zero_feats[:5]) + ('...' if len(zero_feats)>5 else '')
    print(f"  {name:<38} {covered:>5}/{N_SEM}  {covered/N_SEM*100:>7.0f}%   {z_str}")


LOADING DATASETS

LOADING CICIDS-2017
  Loading friday.csv...
    547,557 rows | coverage=42/46 | Ben=288,544 Atk=259,013
  Loading friday_plus.csv...
    547,557 rows | coverage=42/46 | Ben=288,544 Atk=259,013
  Loading monday.csv...
    371,621 rows | coverage=42/46 | Ben=371,621 Atk=0
  Loading monday_plus.csv...
    371,621 rows | coverage=42/46 | Ben=371,621 Atk=0
  Loading thursday.csv...
    362,075 rows | coverage=42/46 | Ben=288,171 Atk=73,904
  Loading thursday_plus.csv...
    362,075 rows | coverage=42/46 | Ben=288,171 Atk=73,904
  Loading tuesday.csv...
    322,078 rows | coverage=42/46 | Ben=315,106 Atk=6,972
  Loading tuesday_plus.csv...
    322,078 rows | coverage=42/46 | Ben=315,106 Atk=6,972
  Loading wednesday.csv...
    496,640 rows | coverage=42/46 | Ben=319,119 Atk=177,521
  Loading wednesday_plus.csv...
    496,640 rows | coverage=42/46 | Ben=319,119 Atk=177,521
  Total: 4,199,942 | Ben=3,165,122 Atk=1,034,820
  Subsampled 4,199,942 -> 500,000 (ben=376,805 atk=123

In [7]:
(X_train_raw, y_train_raw, ctx_train_raw, ds_train_raw,
 X_test_raw,  y_test_raw,  ctx_test_raw,  ds_test_raw) = loader.combine()

N_FEATURES = N_SEM
# NOTE: pkt_count_total derivation has been moved to combine() pre-scaling.
# It is now correctly computed as fwd+bwd on raw counts before RobustScaler.
print("pkt_count_total derived in combine() before scaling — correct")

N_DEVICES  = max(max(d['dev'].max() for d in loader.datasets.values()), 0) + 1
N_DATASETS = len(loader.datasets)
N_DEV_CATS = Config.N_DEV_CATS
N_DS_SRC       = N_DATASETS              # actual datasets loaded
Config.N_DS_SRC = N_DS_SRC              # keep Config in sync
print(f"N_FEATURES={N_FEATURES}  N_DATASETS={N_DATASETS}  DS_NAMES={DS_NAMES}")

def create_sequences(X, y, ctx, ds, window, stride):
    """
    Sliding-window sequencing — vectorised via fancy-index gather.

    ~40x faster than the equivalent Python loop (numpy gather vs Python slice).
    Dataset-boundary-safe: windows never span two datasets.

    Label: majority vote (mean > 0.5).
      With ratio=1.0 row-balance: p(row=atk)≈0.5 → p(win=atk)≈43% (healthy).

    Temporal order preserved within each dataset chunk.
    """
    n_feat = X.shape[1]
    n_ctx  = ctx.shape[1]

    # ── Pass 1: count windows & verify ───────────────────────────────────────
    d_ids = np.unique(ds)
    counts = {}
    for d_id in d_ids:
        n_rows = int((ds == d_id).sum())
        counts[d_id] = max(0, (n_rows - window) // stride + 1) if n_rows >= window else 0
    total = sum(counts.values())
    if total == 0:
        raise ValueError(
            f"No sequences produced — dataset too small for window={window}, stride={stride}. "
            f"Per-dataset row counts: { {d: int((ds==d).sum()) for d in d_ids} }")

    # ── Pass 2: pre-allocate ──────────────────────────────────────────────────
    Xs  = np.empty((total, window, n_feat), dtype=np.float32)
    ys  = np.empty(total,           dtype=np.int32)
    cs  = np.empty((total, n_ctx),  dtype=np.int32)
    dss = np.empty(total,           dtype=np.int32)

    # ── Pass 3: vectorised gather (no Python inner loop) ─────────────────────
    ptr = 0
    for d_id in d_ids:
        n_seq = counts[d_id]
        if n_seq == 0:
            continue
        m = (ds == d_id)
        Xd = X[m]; yd = y[m]; cd = ctx[m]

        # Build index matrix: shape (n_seq, window)
        # Row i starts at i*stride; column j is offset j within window
        start_idx = np.arange(n_seq, dtype=np.int32) * stride        # (n_seq,)
        col_idx   = np.arange(window, dtype=np.int32)                  # (window,)
        idx2d     = start_idx[:, None] + col_idx[None, :]              # (n_seq, window)

        # Vectorised gather — single numpy op (C-level speed)
        Xs[ptr:ptr+n_seq]  = Xd[idx2d]                                 # (n_seq, window, n_feat)
        ys[ptr:ptr+n_seq]  = (yd[idx2d].mean(axis=1) > 0.5).astype(np.int32)
        cs[ptr:ptr+n_seq]  = cd[start_idx]                             # ctx from window start
        dss[ptr:ptr+n_seq] = d_id
        ptr += n_seq

    assert ptr == total, f"Sequence count mismatch: ptr={ptr}, total={total}"
    return Xs, ys, cs, dss

def _cap(X, y, c, d, mx, tag):
    """Cap to mx sequences while enforcing 1:1 class balance."""
    if len(y) <= mx:
        ben = int((y==0).sum()); atk = int((y==1).sum())
        print(f"  [{tag}] {len(y):,} seqs (ben={ben:,} {ben/len(y)*100:.1f}% | atk={atk:,} {atk/len(y)*100:.1f}%)")
        return X, y, c, d
    rng = np.random.RandomState(42)
    ben_idx = np.where(y==0)[0]; atk_idx = np.where(y==1)[0]
    # Enforce strict 1:1 at sequence level up to mx/2 per class
    n_each = mx // 2
    nb = min(len(ben_idx), n_each)
    na = min(len(atk_idx), mx - nb)
    # If one class is very small, let the other fill the cap
    if nb < n_each: na = min(len(atk_idx), mx - nb)
    if na < n_each: nb = min(len(ben_idx), mx - na)
    keep = np.concatenate([rng.choice(ben_idx, nb, replace=False),
                           rng.choice(atk_idx, na, replace=False)])
    rng.shuffle(keep)
    print(f"  [{tag}] Capped {len(y):,} -> {len(keep):,} "
          f"(ben={nb:,} {nb/len(keep)*100:.1f}% | atk={na:,} {na/len(keep)*100:.1f}%)")
    return X[keep], y[keep], c[keep], d[keep]

print("\nCreating sliding-window sequences...")
X_train_s, y_train_s, ctx_train_s, ds_train_s = create_sequences(
    X_train_raw, y_train_raw, ctx_train_raw, ds_train_raw,
    Config.WINDOW_SIZE, Config.STRIDE)
X_test_s, y_test_s, ctx_test_s, ds_test_s = create_sequences(
    X_test_raw, y_test_raw, ctx_test_raw, ds_test_raw,
    Config.WINDOW_SIZE, Config.STRIDE)

print(f"  Raw sequences — Train: {len(y_train_s):,} "
      f"(atk={int((y_train_s==1).sum()):,} {(y_train_s==1).mean()*100:.1f}%) | "
      f"Test: {len(y_test_s):,} "
      f"(atk={int((y_test_s==1).sum()):,} {(y_test_s==1).mean()*100:.1f}%)")

X_train, y_train, ctx_train, ds_train = _cap(
    X_train_s, y_train_s, ctx_train_s, ds_train_s, Config.MAX_TRAIN_SEQ, "train")
X_test, y_test, ctx_test, ds_test = _cap(
    X_test_s, y_test_s, ctx_test_s, ds_test_s, Config.MAX_TEST_SEQ, "test")

# REVISION (R1-m1): validation windows, built ONLY from the validation rows
# (windows never span train/val/test). Used for early stopping and checkpoint
# selection; never for reported metrics.
X_val_s, y_val_s, ctx_val_s, ds_val_s = create_sequences(
    loader.X_val_raw, loader.y_val_raw, loader.ctx_val_raw, loader.ds_val_raw,
    Config.WINDOW_SIZE, Config.STRIDE)
X_val, y_val, ctx_val, ds_val = _cap(
    X_val_s, y_val_s, ctx_val_s, ds_val_s, Config.MAX_VAL_SEQ, "val")
del X_val_s, y_val_s, ctx_val_s, ds_val_s

for _v in ['X_train_raw','X_test_raw','X_train_s','X_test_s']:
    try: exec(f'del {_v}')
    except: pass
gc.collect()
print(f"\nFinal shapes — Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Train: ben={int((y_train==0).sum()):,} ({(y_train==0).mean()*100:.1f}%)  "
      f"atk={int((y_train==1).sum()):,} ({(y_train==1).mean()*100:.1f}%)")
print(f"  Test:  ben={int((y_test==0).sum()):,} ({(y_test==0).mean()*100:.1f}%)  "
      f"atk={int((y_test==1).sum()):,} ({(y_test==1).mean()*100:.1f}%)")

# Sanity checks
assert X_train.dtype == np.float32, "X_train not float32"
assert X_test.dtype  == np.float32, "X_test not float32"
assert X_train.shape[1] == Config.WINDOW_SIZE, f"Wrong window: {X_train.shape[1]}"
assert X_train.shape[2] == N_FEATURES, f"Wrong features: {X_train.shape[2]}"
assert (y_train==1).mean() > 0.10, f"Too few attack seqs in train: {(y_train==1).mean():.3f}"
assert (y_test==1).mean()  > 0.10, f"Too few attack seqs in test:  {(y_test==1).mean():.3f}"
assert (y_train==1).mean() < 0.90, f"Too many attack seqs in train: {(y_train==1).mean():.3f}"
assert (y_test==1).mean()  < 0.90, f"Too many attack seqs in test:  {(y_test==1).mean():.3f}"
assert not np.isnan(X_train).any(), "NaN in X_train after scaling"
assert not np.isnan(X_test).any(),  "NaN in X_test after scaling"
assert X_val.shape[1:] == X_train.shape[1:], "Val window/feature shape mismatch"
assert not np.isnan(X_val).any(),   "NaN in X_val after scaling"
if not (0.05 < (y_val==1).mean() < 0.95):
    print(f"  WARNING: validation windows imbalanced (attack {(y_val==1).mean()*100:.1f}%)")
assert (np.abs(X_train) <= 10.01).all(), "X_train values outside [-10,10] clip range"
print("All assertions passed — data is clean and balanced")
print(f"  X_train: {X_train.shape} | y_train attack%: {(y_train==1).mean()*100:.1f}%")
print(f"  X_test:  {X_test.shape}  | y_test  attack%: {(y_test==1).mean()*100:.1f}%")
print(f"  X_val:   {X_val.shape}  | y_val   attack%: {(y_val==1).mean()*100:.1f}%  (early stopping only)")



COMBINE & PREPROCESS
  CICIDS-2017: 246,390 rows (ben=123,195 atk=123,195 ratio=1.0:1) est_sequences~61,590
  CIC-BCCC-NRC-TabularIoTAttacks-2024: 57,814 rows (ben=28,907 atk=28,907 ratio=1.0:1) est_sequences~14,446
  CICIoMT-2024: 157,624 rows (ben=78,812 atk=78,812 ratio=1.0:1) est_sequences~39,399
  UNSW-NB15: 186,000 rows (ben=93,000 atk=93,000 ratio=1.0:1) est_sequences~46,493

  Pre-split combined: 647,828 rows | ben=323,914 (50.0%) | atk=323,914 (50.0%)
  Split mode: CHRONOLOGICAL (per-dataset train/val/test = 70/10/20% by row order)
  Train: 453,480 | ben=226,545 (50.0%) | atk=226,935 (50.0%)
  Val:   64,782 | ben=32,412 (50.0%) | atk=32,370 (50.0%)
  Test:  129,566 | ben=64,957 (50.1%) | atk=64,609 (49.9%)
    val rows CICIDS-2017                           : 24,639 (attack 50.2%)
    val rows CIC-BCCC-NRC-TabularIoTAttacks-2024   : 5,781 (attack 49.9%)
    val rows CICIoMT-2024                          : 15,762 (attack 49.9%)
    val rows UNSW-NB15                            

In [8]:
_NW = 2 if os.path.exists('/kaggle') else 0

class BotnetDataset(Dataset):
    def __init__(self, X, y, ctx):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.ctx = torch.LongTensor(ctx)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return {'sequence': self.X[i], 'label': self.y[i], 'context': self.ctx[i]}

def make_loaders(Xtr, ytr, ctr, Xte, yte, cte, bs=None):
    if bs is None: bs = Config.BATCH_SIZE
    pw = (_NW > 0)
    # NO WeightedRandomSampler — use plain shuffle.
    # Class balance handled by FocalLoss alpha weighting.
    # This ensures BatchNorm running stats match the actual data distribution
    # (same benign/attack ratio in train AND test), so BN works correctly
    # without hacks like _reset_bn_stats.
    tr = DataLoader(BotnetDataset(Xtr, ytr, ctr), batch_size=bs,
                    shuffle=True, num_workers=_NW,
                    pin_memory=torch.cuda.is_available(), drop_last=True, persistent_workers=pw)
    te = DataLoader(BotnetDataset(Xte, yte, cte), batch_size=bs, shuffle=False,
                    num_workers=_NW, pin_memory=torch.cuda.is_available(), persistent_workers=pw)
    return tr, te

def make_eval_loader(X, y, ctx, bs=None):
    """Non-shuffled loader for a validation or test set."""
    if bs is None: bs = Config.BATCH_SIZE
    return DataLoader(BotnetDataset(X, y, ctx), batch_size=bs, shuffle=False,
                      num_workers=_NW, pin_memory=torch.cuda.is_available(),
                      persistent_workers=(_NW > 0))

train_loader, test_loader = make_loaders(X_train, y_train, ctx_train,
                                          X_test, y_test, ctx_test)
val_loader = make_eval_loader(X_val, y_val, ctx_val)   # REVISION (R1-m1)
print(f"DataLoaders: {len(train_loader)} train | {len(val_loader)} val | {len(test_loader)} test batches")



DataLoaders: 221 train | 32 val | 64 test batches


In [9]:
class DepthwiseSepConv1d(nn.Module):
    def __init__(self, inc, outc, ks=3, pad=1):
        super().__init__()
        self.dw = nn.Conv1d(inc, inc, ks, padding=pad, groups=inc, bias=False)
        self.pw = nn.Conv1d(inc, outc, 1, bias=False)
        self.bn = nn.BatchNorm1d(outc)
    def forward(self, x): return F.relu(self.bn(self.pw(self.dw(x))))

class SEBlock1d(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                nn.Linear(ch, max(ch//r, 4)), nn.ReLU(),
                                nn.Linear(max(ch//r, 4), ch), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x).unsqueeze(-1)

class ResConvBlock(nn.Module):
    def __init__(self, inc, outc):
        super().__init__()
        self.c1 = DepthwiseSepConv1d(inc, outc); self.c2 = DepthwiseSepConv1d(outc, outc)
        self.se = SEBlock1d(outc)
        self.skip = nn.Conv1d(inc, outc, 1, bias=False) if inc != outc else nn.Identity()
        self.bn = nn.BatchNorm1d(outc) if inc != outc else nn.Identity()
    def forward(self, x):
        out = self.se(self.c2(self.c1(x))); sk = self.bn(self.skip(x))
        if out.shape[-1] != sk.shape[-1]: sk = F.adaptive_avg_pool1d(sk, out.shape[-1])
        return F.relu(out + sk)

# ── CB-GAF: Cross-Branch Gated Attention Fusion (logit-level mixture) ─────────
# GATED LATE FUSION. Each active branch owns a classifier head producing its own
# logits; a softmax "attention over branches" gate (conditioned on all branch
# representations) mixes those logits per sample:
#     logits = Σ_i  a_i(x) · head_i(rep_i),    a(x) = softmax(gate([rep_1..rep_n]))
# Because the gate can place all its mass on one branch, the fused predictor can
# represent ANY single-branch predictor EXACTLY. So under a shared training budget
# the full model matches-or-beats its best single branch instead of being diluted
# by a shared concat head (the failure mode of feature-level fusion where a weak
# branch injects noise into one classifier). Per-sample routing additionally lets
# the fusion EXCEED both branches where they are complementary. Returns the mixed
# logits, a gated feature mixture (for the CORAL loss + aux decoder), and the gate
# weights (for diagnostics).
class CrossBranchGatedAttention(nn.Module):
    def __init__(self, dims, nc, hid=128, do=0.1):
        super().__init__()
        self.n = len(dims); self.feat_dim = hid
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d, hid), nn.BatchNorm1d(hid), nn.GELU(),
                          nn.Dropout(do), nn.Linear(hid, nc)) for d in dims])
        self.fp = nn.ModuleList([nn.Linear(d, hid) for d in dims])
        if self.n > 1:
            self.gate = nn.Sequential(nn.Linear(sum(dims), hid), nn.GELU(),
                                       nn.Dropout(do), nn.Linear(hid, self.n))

    def forward(self, reps):
        logits_i = torch.stack([h(r) for h, r in zip(self.heads, reps)], 1)   # (B,n,nc)
        if self.n > 1:
            a = F.softmax(self.gate(torch.cat(reps, -1)), -1)                  # (B,n)
        else:
            a = torch.ones(reps[0].size(0), 1, device=reps[0].device, dtype=reps[0].dtype)
        logits = (a.unsqueeze(-1) * logits_i).sum(1)                           # (B,nc)
        feat = sum(a[:, i:i+1] * F.gelu(self.fp[i](reps[i])) for i in range(self.n))  # (B,hid)
        return logits, feat, a, logits_i

class TCHNetV2(nn.Module):
    """
    TCH-Net v2 — Temporal (T) + Statistical (H) branches, GATED LATE FUSION.
    The identity Contextual (C) branch remains REMOVED: dataset-identity
    conditioning does not exist for an unseen LODO domain and was the mechanism
    that failed to transfer. Cross-domain generalisation is handled by
    (i) CORAL source-domain feature alignment during training and
    (ii) transductive test-time BatchNorm adaptation at LODO eval.

    Fusion (Sec. 4.7): CB-GAF is a logit-level mixture over branch experts
    (each branch has its own head; a learned gate mixes their logits), trained
    with DEEP SUPERVISION: an auxiliary CE on each expert's own logits
    (weight Config.DEEP_SUP_WT) alongside the fused CE. The per-expert loss
    prevents mixture collapse onto the fastest-training expert (the T-branch
    fits the training set quickest and otherwise monopolises the gate): every
    expert is trained to stand alone, and the gate SELECTS — it can recover
    any single branch exactly, so the fused model matches-or-beats its best
    trained branch, with per-sample routing to exceed both where they are
    complementary.

    Ablation flags (all default True → the full model):
      Branch ablation : use_t, use_h            (C is gone; configs are T+H/T/H)
      Novelty ablation: use_ms, use_cbgaf, use_aux
        - use_ms   : 3-path multi-scale T-branch (True) vs Path-1 only (False)
        - use_cbgaf: gated late fusion (True) vs LayerNorm(concat)+shared head (False)
        - use_aux  : auxiliary reconstruction decoder on/off
      CORAL (use_coral) is a TRAINING-side loss, toggled in the training loop.
    raw_proj (Sec. 4.8) is the direct statistical pathway, folded into the H
    representation so the H expert and the H-only ablation share identical input.
    """
    def __init__(self, nf, ws, ed=None, cc=None, nc=None,
                 gh=None, gl=None, ah=None, do=None, cd=None,
                 use_t=True, use_h=True,
                 use_ms=True, use_cbgaf=True, use_aux=True):
        super().__init__()
        nc=nc or Config.N_CLASSES; self.nc=nc
        ed=ed or Config.EMBED_DIM; cc=cc or Config.CONV_CH
        gh=gh or Config.GRU_HIDDEN; gl=gl or Config.GRU_LAYERS
        ah=ah or Config.ATTN_HEADS; do=do if do is not None else Config.DROPOUT
        cd=cd or Config.CBGAF_DIM
        assert use_t or use_h, "at least one of T/H must be active"
        self.use_t=use_t; self.use_h=use_h
        self.use_ms=use_ms; self.use_cbgaf=use_cbgaf; self.use_aux=use_aux
        self._nf=nf

        self.feat_proj = nn.Sequential(
            nn.Linear(nf, nf*2), nn.LayerNorm(nf*2), nn.GELU(),
            nn.Dropout(do*0.5), nn.Linear(nf*2, nf), nn.LayerNorm(nf))

        # ── T-branch ──
        s1 = gh*2
        if use_ms:
            td = s1 + (gh//2)*2 + 128
        else:
            td = s1
        self._td = td
        if use_t:
            ls, ic = [], nf
            for i, oc in enumerate(cc):
                ls.append(ResConvBlock(ic, oc))
                ls.append(nn.MaxPool1d(2,2) if i < len(cc)-1 else nn.AdaptiveAvgPool1d(8))
                ic = oc
            self.t_conv = nn.Sequential(*ls)
            self.t_gru1 = nn.GRU(cc[-1], gh, gl, batch_first=True, bidirectional=True,
                                  dropout=do if gl>1 else 0)
            if use_ms:
                self.t_down = nn.Sequential(nn.Conv1d(nf, cc[0], 3, stride=2, padding=1, bias=False),
                                             nn.BatchNorm1d(cc[0]), nn.ReLU())
                self.t_gru2 = nn.GRU(cc[0], gh//2, 1, batch_first=True, bidirectional=True)
                t_dm = 128
                self.t_proj = nn.Linear(nf, t_dm)
                self.t_pos  = nn.Parameter(torch.randn(1, ws, t_dm) * 0.02)
                _tel = nn.TransformerEncoderLayer(t_dm, 8, t_dm*4, do,
                                                   batch_first=True, norm_first=True)
                self.t_cls  = nn.Parameter(torch.zeros(1, 1, t_dm))
                self.t_enc  = nn.TransformerEncoder(_tel, 2)
            h_ = ah
            while td % h_ != 0 and h_ > 1: h_ -= 1
            self.t_ln  = nn.LayerNorm(td)
            self.t_mha = nn.MultiheadAttention(td, h_, batch_first=True, dropout=do)

        # ── H-branch (statistical MLP + direct raw pathway, folded together) ──
        # h_rep = [h_mlp(mean) (64) ‖ raw_proj(mean) (64)] = 128, so the H expert
        # and the H-only ablation receive the identical statistical input.
        hd = 128; self._hd = hd
        if use_h:
            self.h_mlp = nn.Sequential(nn.Linear(nf,128), nn.BatchNorm1d(128), nn.GELU(),
                                        nn.Dropout(do), nn.Linear(128,64), nn.BatchNorm1d(64),
                                        nn.GELU(), nn.Dropout(do))
            self.raw_proj = nn.Sequential(nn.Linear(nf, 64), nn.BatchNorm1d(64), nn.GELU())

        # ── Fusion over the active branches ──
        _dims = []
        if use_t: _dims.append(td)
        if use_h: _dims.append(hd)
        self._branches = [b for b, on in (('T', use_t), ('H', use_h)) if on]
        self._dims = _dims

        if use_cbgaf:
            # Gated late fusion (mixture over branch logits)
            self.cbgaf = CrossBranchGatedAttention(_dims, nc, hid=cd, do=do)
            fused_dim = self.cbgaf.feat_dim
        else:
            # Ablation baseline: LayerNorm(concat) → shared residual MLP head
            fd = sum(_dims)
            self.fln  = nn.LayerNorm(fd)
            self.clf1 = nn.Sequential(nn.Linear(fd,256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(do))
            self.clf2 = nn.Sequential(nn.Linear(256,128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(do))
            self.clf_res = nn.Linear(fd,128)
            self.clf_out = nn.Linear(128, nc)
            fused_dim = fd
        self._fd = fused_dim

        if use_aux:
            self.aux = nn.Sequential(nn.Linear(fused_dim, 64), nn.GELU(), nn.Linear(64, nf))

    def forward(self, x, ctx=None, return_features=False):
        B = x.shape[0]
        x = x + self.feat_proj(x)
        xt = x.transpose(1, 2)

        reps = []
        if self.use_t:
            c1 = self.t_conv(xt).transpose(1, 2)
            g1, _ = self.t_gru1(c1)
            if self.use_ms:
                c2 = self.t_down(xt).transpose(1, 2)
                g2, _ = self.t_gru2(c2)
                g2e = F.adaptive_avg_pool1d(g2.transpose(1,2), g1.size(1)).transpose(1,2)
                t_tok = self.t_proj(x) + self.t_pos[:, :x.size(1), :]
                cls_tok = self.t_cls.expand(B, -1, -1)
                t_in = torch.cat([cls_tok, t_tok], dim=1)
                t_out = self.t_enc(t_in)[:, 1:, :]
                t_e = F.adaptive_avg_pool1d(t_out.transpose(1,2), g1.size(1)).transpose(1,2)
                gc_ = torch.cat([g1, g2e, t_e], dim=-1)
            else:
                gc_ = g1
            ao, _ = self.t_mha(self.t_ln(gc_), self.t_ln(gc_), self.t_ln(gc_))
            reps.append(ao.mean(1))                                  # (B, td)
        if self.use_h:
            xm = x.mean(1)
            reps.append(torch.cat([self.h_mlp(xm), self.raw_proj(xm)], dim=-1))  # (B, 128)

        if self.use_cbgaf:
            logits, fused, _gate, _bl = self.cbgaf(reps)
            # stash for deep supervision (training loop) + gate diagnostics
            self.branch_logits = _bl          # (B, n_branches, nc)
            self.gate_weights  = _gate.detach()                 # gated late fusion
        else:
            self.branch_logits = None
            self.gate_weights  = None
            fused = self.fln(torch.cat(reps, -1))                   # concat baseline
            h2 = self.clf2(self.clf1(fused)) + self.clf_res(fused)
            logits = self.clf_out(h2)

        recon = self.aux(fused) if self.use_aux else torch.zeros(B, self._nf, device=x.device, dtype=x.dtype)
        if return_features: return logits, recon, fused
        return logits, recon

def make_tch_v2(): return TCHNetV2(N_FEATURES, Config.WINDOW_SIZE, nc=Config.N_CLASSES)

def _get_temporal_attention(self, x, ctx=None):
    """Return MHA attention weights (B, n_heads, 8, 8) for viz — no grad."""
    B = x.shape[0]
    x2 = x + self.feat_proj(x)
    xt = x2.transpose(1, 2)
    c1  = self.t_conv(xt).transpose(1, 2)
    g1, _ = self.t_gru1(c1)
    c2 = self.t_down(xt).transpose(1, 2)
    g2, _ = self.t_gru2(c2)
    g2e = F.adaptive_avg_pool1d(g2.transpose(1,2), g1.size(1)).transpose(1,2)
    t_tok = self.t_proj(x2) + self.t_pos[:, :x2.size(1), :]
    cls_tok = self.t_cls.expand(B, -1, -1)
    t_in  = torch.cat([cls_tok, t_tok], dim=1)
    t_out = self.t_enc(t_in)[:, 1:, :]
    t_e   = F.adaptive_avg_pool1d(t_out.transpose(1,2), g1.size(1)).transpose(1,2)
    gc    = torch.cat([g1, g2e, t_e], dim=-1)
    q = k = v = self.t_ln(gc)
    _, attn_w = self.t_mha(q, k, v, need_weights=True, average_attn_weights=False)
    return attn_w

TCHNetV2.get_temporal_attention = _get_temporal_attention

m = make_tch_v2().to(device)
np_ = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"TCH-Net v2 (T+H, gated late fusion): {np_:,} params | T={m._td}d H={m._hd}d -> CB-GAF(gated MoE, {m._fd}d)")
del m


TCH-Net v2 (T+H, gated late fusion): 2,385,940 params | T=512d H=128d -> CB-GAF(gated MoE, 128d)


In [10]:
_amp_en = torch.cuda.is_available()
_amp_dev = 'cuda' if _amp_en else 'cpu'
# Fresh scaler per train_full — avoids degraded fp16 state carried across seeds
def _make_scaler(): return torch.amp.GradScaler(enabled=_amp_en)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2., alpha=None, ls=0.):
        super().__init__(); self.gamma=gamma; self.alpha=alpha; self.ls=ls
    def forward(self, lg, tgt):
        lf = lg.float(); af = self.alpha.float() if self.alpha is not None else None
        ce = F.cross_entropy(lf, tgt, weight=af, reduction='none', label_smoothing=self.ls)
        pt = torch.exp(-ce.clamp(max=80)).clamp(1e-7, 1-1e-7)
        return ((1-pt)**self.gamma * ce).mean()

def get_sched(opt, wu, total):
    def fn(ep):
        if ep < wu: return (ep+1)/wu
        prog = (ep-wu)/max(total-wu,1)
        return max(0.05, 0.5*(1+np.cos(np.pi*prog)))
    return optim.lr_scheduler.LambdaLR(opt, fn)

def make_criterion(y=None):
    if y is None: y = y_train
    cc = np.bincount(y)
    # Alpha weighting is the ONLY class balance mechanism now
    # (WeightedRandomSampler removed → plain shuffle).
    # This keeps BN running stats matching real distribution.
    w = torch.FloatTensor([1., cc[0]/max(cc[1],1)]).to(device)
    w = w / w.sum() * 2   # normalize so mean weight = 1
    return FocalLoss(Config.FOCAL_GAMMA, w, Config.LABEL_SMOOTH)

def train_ep_v2(mdl, dl, crit, opt, aw, scaler, coral_wt=0.0):
    mdl.train(); tl=c=t=0; mse=nn.MSELoss()
    for b in tqdm(dl, desc="  batches", leave=False):
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        opt.zero_grad()
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            lg, rc, ft = mdl(s, ctx, return_features=True)
            loss = crit(lg,l) + aw*mse(rc, s.mean(1))
            # Deep supervision: auxiliary CE on each expert's OWN logits.
            # Without it the gate collapses onto the fastest-training expert
            # (T fits train fastest -> T+H degenerates to T while a solo-trained
            # H generalises better). Per-expert CE trains every expert to stand
            # alone; the gate then selects between well-trained experts.
            _bl = getattr(mdl, 'branch_logits', None)
            _dsw = getattr(Config, 'DEEP_SUP_WT', 0.0)
            if _bl is not None and _dsw > 0:
                loss = loss + _dsw * sum(crit(_bl[:, _i], l)
                                         for _i in range(_bl.size(1))) / _bl.size(1)
            # CORAL: align fused-feature covariances across the SOURCE domains
            # present in the batch (ctx[:,0] = dataset id). Domain-invariant
            # features are the lever for cross-dataset (LODO) transfer.
            if coral_wt > 0:
                loss = loss + coral_wt * coral_loss(ft, ctx[:, 0])
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)  # tighter for transformer stability
        scaler.step(opt); scaler.update()
        tl += loss.item(); _, p = lg.max(1); c += (p==l).sum().item(); t += l.size(0)
    return tl/len(dl), c/t

def train_ep_base(mdl, dl, crit, opt, scaler):
    mdl.train(); tl=c=t=0
    for b in tqdm(dl, desc="  batches", leave=False):
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        opt.zero_grad()
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            out = mdl(s, ctx); loss = crit(out, l)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), 2.0)
        scaler.step(opt); scaler.update()
        tl += loss.item(); _, p = out.max(1); c += (p==l).sum().item(); t += l.size(0)
    return tl/len(dl), c/t

@torch.no_grad()
def evaluate(mdl, dl, crit, is_v2=False, verbose=True):
    mdl.eval()
    ps=[]; ls=[]; pbs=[]; tl=0
    _dl = tqdm(dl, desc="  eval", leave=False) if verbose else dl
    for b in _dl:
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            out = mdl(s, ctx); lg = out[0] if is_v2 else out
            tl += crit(lg.float(), l).item()
        pr = F.softmax(lg.float(),-1); _, pd = lg.max(1)
        ps.extend(pd.cpu().numpy()); ls.extend(l.cpu().numpy()); pbs.extend(pr[:,1].cpu().numpy())
    yt=np.array(ls); yp=np.array(ps); ypr=np.array(pbs)
    try: roc=roc_auc_score(yt,ypr)
    except: roc=0.
    try: pp,pr_,_=precision_recall_curve(yt,ypr); prauc=auc(pr_,pp)
    except: prauc=0.
    try:
        fa,ta,_=roc_curve(yt,ypr)
        fpr99=float(fa[min(np.searchsorted(ta,0.99),len(fa)-1)])
        fpr95=float(fa[min(np.searchsorted(ta,0.95),len(fa)-1)])
    except: fpr99=1.; fpr95=1.
    cr = classification_report(yt,yp,target_names=['Benign','Attack'],output_dict=True,zero_division=0)
    return {'accuracy':accuracy_score(yt,yp),'precision':precision_score(yt,yp,zero_division=0),
            'recall':recall_score(yt,yp,zero_division=0),'f1':f1_score(yt,yp,zero_division=0),
            'roc_auc':roc,'mcc':matthews_corrcoef(yt,yp),'pr_auc':prauc,'fpr_at_tpr99':fpr99,
            'benign_f1':cr['Benign']['f1-score'],'attack_f1':cr['Attack']['f1-score'],
            'fpr_at_tpr95':fpr95,
            'loss':tl/max(len(dl),1),'predictions':yp,'labels':yt,'probabilities':ypr}

def _selection_score(m):
    """Early-stopping criterion on the VALIDATION set: F1 when the validation
       windows contain both classes, otherwise negative validation loss."""
    return m['f1'] if len(np.unique(m['labels'])) > 1 else -m['loss']

def train_full(mdl, trdl, vadl, crit, epochs, lr, wd, es, wu=5,
               is_v2=False, aw=0.1, verbose=True, coral_wt=None, tedl=None):
    """Train with early stopping and best-checkpoint selection on `vadl`, the
       VALIDATION loader (never the test set). After the best checkpoint is
       restored, the returned metrics are computed on `tedl` (test) if given,
       otherwise on `vadl`. `hist` holds per-epoch VALIDATION metrics.
       REVISION (R1-m1): selection previously monitored the test loader."""
    if coral_wt is None: coral_wt = getattr(Config, 'CORAL_WT', 0.0)
    opt = optim.AdamW(mdl.parameters(), lr=lr, weight_decay=wd)
    sch = get_sched(opt, wu, epochs)
    scaler = _make_scaler()
    bf=-np.inf; bs_=None; pat=0; hist=[]
    ep_iter = tqdm(range(epochs), desc="Epochs", leave=True) if verbose else range(epochs)
    for ep in ep_iter:
        if is_v2: tl,_ = train_ep_v2(mdl,trdl,crit,opt,aw,scaler,coral_wt=coral_wt)
        else:     tl,_ = train_ep_base(mdl,trdl,crit,opt,scaler)
        sch.step(); m = evaluate(mdl,vadl,crit,is_v2, verbose=verbose)
        m['train_loss']=tl
        # Strip large per-epoch arrays — only needed in final evaluate call
        _m_hist = {k:v for k,v in m.items() if k not in ('predictions','labels','probabilities')}
        hist.append(_m_hist)
        _score = _selection_score(m)
        is_best = _score > bf
        if verbose:
            # BUG FIX: avoid nested quotes in f-strings (SyntaxError in Python <3.12)
            _f1=m['f1']; _auc=m['roc_auc']; _mcc=m['mcc']
            _bfv=m['benign_f1']; _afv=m['attack_f1']
            btag = " \u2605" if is_best else ""
            if hasattr(ep_iter,'set_postfix'):
                ep_iter.set_postfix(
                    loss="{:.4f}".format(tl), val_f1="{:.4f}".format(_f1),
                    auc="{:.4f}".format(_auc), mcc="{:.4f}".format(_mcc),
                    best="{:.4f}".format(bf))
            tqdm.write("  Ep {:3d}/{:d}  Loss={:.4f}  valF1={:.4f}  valAUC={:.4f}  valMCC={:.4f}"
                       "  BenF1={:.4f}  AtkF1={:.4f}{}".format(
                           ep+1, epochs, tl, _f1, _auc, _mcc, _bfv, _afv, btag))
        if is_best: bf=_score; bs_=copy.deepcopy(mdl.state_dict()); pat=0
        else:
            pat += 1
            if pat >= es:
                if verbose: tqdm.write("  \u23f9 Early stop at epoch {:d}  |  Best val score={:.4f}".format(ep+1,bf))
                break
    if bs_: mdl.load_state_dict(bs_)
    return evaluate(mdl, tedl if tedl is not None else vadl, crit, is_v2), hist

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def multi_seed(model_fn, seeds, epochs, is_v2=False, aw=0.1, wu=None,
               save_sd0=False, coral_wt=None,
               Xtr=None, ytr=None, ctr=None, Xte=None, yte=None, cte=None,
               Xva=None, yva=None, cva=None):
    if Xtr is None: Xtr,ytr,ctr = X_train,y_train,ctx_train
    if Xte is None: Xte,yte,cte = X_test,y_test,ctx_test
    if Xva is None: Xva,yva,cva = X_val,y_val,ctx_val
    all_m = []; all_h = []; _sd0 = None
    for i,s in enumerate(seeds):
        print("  -- Seed {} [{}/{}] --".format(s, i+1, len(seeds)))
        set_seed(s); mdl = model_fn().to(device)
        tl,tel = make_loaders(Xtr,ytr,ctr,Xte,yte,cte)
        vl = make_eval_loader(Xva,yva,cva)
        cr = make_criterion(ytr)
        _wu = wu if wu is not None else Config.WARMUP
        # Select on validation (vl); report on test (tel) — REVISION (R1-m1)
        m,h_ = train_full(mdl,tl,vl,cr,epochs,Config.LR,Config.WD,Config.EARLY_STOP,
                          _wu,is_v2,aw,coral_wt=coral_wt,tedl=tel)
        m['seed'] = s   # REVISION (R1-M1): explicit seed tag for seed-matched pairing
        print("  |  F1={:.4f}  AUC={:.4f}  MCC={:.4f}  PR-AUC={:.4f}".format(
              m['f1'],m['roc_auc'],m['mcc'],m['pr_auc']))
        print("  |  Acc={:.4f}  Prec={:.4f}  Rec={:.4f}".format(
              m['accuracy'],m['precision'],m['recall']))
        print("  |  Benign-F1={:.4f}  Attack-F1={:.4f}  FPR@TPR99={:.4f}  FPR@TPR95={:.4f}".format(
              m['benign_f1'],m['attack_f1'],m['fpr_at_tpr99'],m.get('fpr_at_tpr95',1.)))
        if save_sd0 and i == 0:
            _sd0 = copy.deepcopy(mdl.state_dict())  # seed-42 best weights for analysis
        all_m.append(m); all_h.append(h_); del mdl; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return all_m, all_h, _sd0

def summarise(ml):
    ks=['accuracy','precision','recall','f1','roc_auc','mcc','pr_auc','benign_f1','attack_f1','fpr_at_tpr99','fpr_at_tpr95']
    s={}
    for k in ks:
        v=[m[k] for m in ml if k in m]
        # REVISION (R1-M1): sample SD (ddof=1); previously population SD (ddof=0)
        if v: s[k+'_mean']=float(np.mean(v)); s[k+'_std']=float(np.std(v, ddof=1)) if len(v)>1 else 0.; s[k+'_vals']=v
    s['seeds'] = [m.get('seed') for m in ml]   # aligned with every *_vals list
    # Store concatenated probabilities and labels across all seeds for
    # full operational metric computation (Precision@TPR, TPR@FPR, Brier)
    all_probs  = [m['probabilities'] for m in ml if 'probabilities' in m and len(m['probabilities'])>0]
    all_labels = [m['labels']        for m in ml if 'labels'        in m and len(m['labels'])>0]
    if all_probs and all_labels:
        s['_probs_all']  = np.concatenate(all_probs)
        s['_labels_all'] = np.concatenate(all_labels)
    return s

# ── CORAL: source-domain second-order feature alignment ──────────────────────
def coral_loss(feats, domains):
    """Mean pairwise CORAL distance between per-domain covariances of `feats`.
       feats: (B,d) fused features; domains: (B,) integer dataset ids.
       Returns 0 if <2 domains (or <2 samples/domain) are present in the batch."""
    feats = feats.float()
    uniq = torch.unique(domains)
    if uniq.numel() < 2: return feats.new_zeros(())
    d = feats.size(1); covs = []
    for u in uniq:
        f = feats[domains == u]
        if f.size(0) < 2: continue
        f = f - f.mean(0, keepdim=True)
        covs.append((f.t() @ f) / (f.size(0) - 1))
    if len(covs) < 2: return feats.new_zeros(())
    loss = feats.new_zeros(()); cnt = 0
    for i in range(len(covs)):
        for j in range(i+1, len(covs)):
            loss = loss + ((covs[i] - covs[j])**2).sum() / (4.0 * d * d); cnt += 1
    return loss / max(cnt, 1)

# ── Transductive test-time BatchNorm adaptation (AdaBN, Li et al. 2017) ───────
@torch.no_grad()
def adapt_bn(mdl, loader, max_batches=None):
    """Recompute BN running stats from the (UNLABELED) target-domain data in
       `loader`, then return the model in eval mode. No labels are used. No-op
       for BN-free models (e.g. Transformer-IDS, LayerNorm-only)."""
    bns = [m for m in mdl.modules() if isinstance(m, nn.BatchNorm1d)]
    if not bns:
        mdl.eval(); return mdl
    was_training = mdl.training
    saved = [m.momentum for m in bns]
    for m in bns:
        m.reset_running_stats(); m.momentum = None; m.train()   # cumulative average
    for i, b in enumerate(loader):
        s = b['sequence'].to(device); ctx = b['context'].to(device)
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            mdl(s, ctx)
        if max_batches and (i + 1) >= max_batches: break
    for m, mom in zip(bns, saved): m.momentum = mom; m.eval()
    mdl.eval()
    return mdl

# ── Threshold helpers (data-driven grid, percentile candidates) ──────────────
def best_threshold(probs, labels, n=200):
    cand = np.unique(np.percentile(probs, np.linspace(0, 100, n)))
    bf, bt = 0.0, 0.5
    for t in cand:
        f = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f > bf: bf, bt = f, t
    return bf, bt

def calibrated_f1(probs, labels, seed, cal_frac=0.20):
    """Threshold picked on a disjoint cal slice, F1 reported on the eval slice."""
    if probs is None or labels is None or len(probs) < 20:
        return (float(f1_score(labels, (probs>=0.5).astype(int), zero_division=0))
                if probs is not None and len(probs) else 0.0), 0.5, 0
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(probs)); n_cal = max(10, int(cal_frac*len(probs)))
    ci, ei = perm[:n_cal], perm[n_cal:]
    _, thr = best_threshold(probs[ci], labels[ci])
    return float(f1_score(labels[ei], (probs[ei] >= thr).astype(int), zero_division=0)), thr, n_cal

print("Training infrastructure ready  |  CORAL + test-time BN adaptation enabled")



Training infrastructure ready  |  CORAL + test-time BN adaptation enabled


## Main-run reference values
The temporal cell compares against the main 5-seed run (`tch_summary`). That training is not repeated here, so its means are taken from `tch_summary_ci.json` of the full Kaggle run (10 Oct 2026).

In [11]:
# Main 5-seed run (full revised notebook, Kaggle, 10 Oct 2026) — from tch_summary_ci.json
tch_summary = {'f1_mean': 0.8807, 'roc_auc_mean': 0.9656, 'mcc_mean': 0.7903, 'pr_auc_mean': 0.9572,
               'fpr_at_tpr99_mean': 0.3750, 'fpr_at_tpr95_mean': 0.1840}
print("Reference (main run, 5 seeds):", tch_summary)


Reference (main run, 5 seeds): {'f1_mean': 0.8807, 'roc_auc_mean': 0.9656, 'mcc_mean': 0.7903, 'pr_auc_mean': 0.9572, 'fpr_at_tpr99_mean': 0.375, 'fpr_at_tpr95_mean': 0.184}


## Temporal re-split (3 seeds)

In [12]:
try: _ds_test_bkp = ds_test.copy()
except: _ds_test_bkp = None
print("="*70+"\nTEMPORAL SPLIT\n"+"="*70)

# ── Step 0: Free main training arrays — no longer needed after Cell 24 ────────
# X_train (2.8 GB) + X_test (0.7 GB) sit idle here and are the primary OOM cause.
# DataLoaders (train_loader/test_loader) hold references too — delete both together.
_to_free = ['X_train', 'X_test', 'y_train', 'y_test',
            'ctx_train', 'ctx_test', 'ds_train', 'ds_test',
            'train_loader', 'test_loader',
            'y_train_s', 'y_test_s']
for _v in _to_free:
    try: exec(f'del {_v}')
    except: pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("  Freed main training arrays (~3.5 GB recovered)")

# ── Step 1: Build temporal train/test splits ONE DATASET AT A TIME ────────────
# Each dataset copy is freed immediately after slices are appended, so peak
# extra memory per iteration ≤ one dataset (~0.55 GB) rather than all five.
Xtr_t, ytr_t, ctr_t, dtr_t = [], [], [], []
Xte_t, yte_t, cte_t, dte_t = [], [], [], []
Xva_t, yva_t, cva_t, dva_t = [], [], [], []   # REVISION (R1-m1): validation slice
for di, (name, data) in enumerate(loader.datasets.items()):
    X_ = data['X'].copy(); y_ = data['y'].copy(); c_ = data['ctx'].copy()
    np.nan_to_num(X_, copy=False, nan=0., posinf=1e6, neginf=-1e6)
    ben = np.where(y_==0)[0]; atk = np.where(y_==1)[0]
    # FIX: the original version only ever capped attack DOWN toward benign
    # (`if len(atk) > mx: subsample atk`), with no symmetric step when
    # benign was the naturally larger class — so any dataset where benign
    # already dominated stayed skewed, instead of getting the same strict
    # 1:1 balance every other cell in this notebook uses. This is exactly
    # why the temporal test set came out ~87%/13% benign/attack instead of
    # ~50/50 like the random-split pipeline, confounding the "is performance
    # similar under temporal vs random split" comparison with "is performance
    # similar under a totally different class balance" at the same time.
    # Also fixed: the original used the bare global `np.random.choice`
    # (unseeded, depends on whatever ran earlier in the notebook) instead of
    # a seeded RandomState like every other balance step here uses.
    _rng_temporal = np.random.RandomState(42 + di)
    n_min_t = min(len(ben), len(atk))
    mx = int(Config.TARGET_ATK_BEN_RATIO * n_min_t)
    if len(ben) > n_min_t: ben = _rng_temporal.choice(ben, n_min_t, replace=False)
    if len(atk) > mx:      atk = _rng_temporal.choice(atk, mx,      replace=False)
    keep = np.sort(np.concatenate([ben, atk]))
    X_, y_, c_ = X_[keep], y_[keep], c_[keep]
    # REVISION (R1-m1): train | val | test = 70 | 10 | 20 % by row order
    n = len(y_); n_trva = int(n * 0.8); n_va = int(n * Config.VAL_FRAC); n_tr = n_trva - n_va
    Xtr_t.append(X_[:n_tr].copy()); ytr_t.append(y_[:n_tr].copy()); ctr_t.append(c_[:n_tr].copy())
    dtr_t.append(np.full(n_tr, di, dtype=np.int32))
    Xva_t.append(X_[n_tr:n_trva].copy()); yva_t.append(y_[n_tr:n_trva].copy()); cva_t.append(c_[n_tr:n_trva].copy())
    dva_t.append(np.full(n_trva - n_tr, di, dtype=np.int32))
    Xte_t.append(X_[n_trva:].copy()); yte_t.append(y_[n_trva:].copy()); cte_t.append(c_[n_trva:].copy())
    dte_t.append(np.full(n - n_trva, di, dtype=np.int32))
    print(f"  {name}: train={n_tr:,} val={n_trva-n_tr:,} test={n-n_trva:,}")
    del X_, y_, c_  # free the per-dataset working copy immediately

# ── Step 2: Stack — delete each list the moment its stacked array is ready ────
# Both the list and the stacked result coexist briefly during vstack; deleting
# the list right after halves the transient spike (saves ~2 GB at peak).
Xtr_tc = np.vstack(Xtr_t);  del Xtr_t
ytr_tc = np.hstack(ytr_t);  del ytr_t
ctr_tc = np.vstack(ctr_t);  del ctr_t
dtr_tc = np.hstack(dtr_t);  del dtr_t
gc.collect()
Xte_tc = np.vstack(Xte_t);  del Xte_t
yte_tc = np.hstack(yte_t);  del yte_t
cte_tc = np.vstack(cte_t);  del cte_t
dte_tc = np.hstack(dte_t);  del dte_t
gc.collect()
Xva_tc = np.vstack(Xva_t);  del Xva_t
yva_tc = np.hstack(yva_t);  del yva_t
cva_tc = np.vstack(cva_t);  del cva_t
dva_tc = np.hstack(dva_t);  del dva_t

# ── Step 3: Scale ─────────────────────────────────────────────────────────────
sc_t = RobustScaler(quantile_range=(5, 95))
Xtr_tc = np.clip(sc_t.fit_transform(Xtr_tc), -10, 10).astype(np.float32)
Xte_tc = np.clip(sc_t.transform(Xte_tc),     -10, 10).astype(np.float32)
Xva_tc = np.clip(sc_t.transform(Xva_tc),     -10, 10).astype(np.float32)

# ── Step 4: Sequence creation — free row arrays as soon as sequences exist ────
# Xtr_tc (~1.96 GB) and Xtr_ts (~2.8 GB) must not coexist longer than necessary.
Xtr_ts, ytr_ts, ctr_ts, _ = create_sequences(
    Xtr_tc, ytr_tc, ctr_tc, dtr_tc, Config.WINDOW_SIZE, Config.STRIDE)
del Xtr_tc, ytr_tc, ctr_tc, dtr_tc; gc.collect()  # free ~2 GB before test sequences

Xte_ts, yte_ts, cte_ts, _ = create_sequences(
    Xte_tc, yte_tc, cte_tc, dte_tc, Config.WINDOW_SIZE, Config.STRIDE)
del Xte_tc, yte_tc, cte_tc, dte_tc; gc.collect()  # free ~0.5 GB before cap
Xva_ts, yva_ts, cva_ts, _ = create_sequences(
    Xva_tc, yva_tc, cva_tc, dva_tc, Config.WINDOW_SIZE, Config.STRIDE)
del Xva_tc, yva_tc, cva_tc, dva_tc; gc.collect()

# ── Step 5: Cap & train ───────────────────────────────────────────────────────
Xtr_ts, ytr_ts, ctr_ts, _ = _cap(Xtr_ts, ytr_ts, ctr_ts,
    np.zeros(len(ytr_ts), dtype=np.int32), Config.MAX_TRAIN_SEQ, "t_tr")
Xte_ts, yte_ts, cte_ts, _ = _cap(Xte_ts, yte_ts, cte_ts,
    np.zeros(len(yte_ts), dtype=np.int32), Config.MAX_TEST_SEQ, "t_te")
Xva_ts, yva_ts, cva_ts, _ = _cap(Xva_ts, yva_ts, cva_ts,
    np.zeros(len(yva_ts), dtype=np.int32), Config.MAX_VAL_SEQ, "t_va")

# ── Multi-seed temporal split (one seed not enough) ────────────────────
# Using Config.FAST_SEEDS = [42, 123, 456] — 3 seeds to establish that the
# chronological split results are stable, not a lucky single-run artefact.
print("\n  Training TCH-Net v2 on temporal split (3 seeds)...")
_tm_seed_metrics = []
for _ts_seed in Config.FAST_SEEDS:
    set_seed(_ts_seed)
    _tm = make_tch_v2().to(device)
    _ttl, _tte = make_loaders(Xtr_ts, ytr_ts, ctr_ts, Xte_ts, yte_ts, cte_ts)
    _tva = make_eval_loader(Xva_ts, yva_ts, cva_ts)
    _tc = make_criterion(ytr_ts)
    _tm_m, _ = train_full(_tm, _ttl, _tva, _tc, Config.EPOCHS, Config.LR, Config.WD,
                           Config.EARLY_STOP, Config.WARMUP, is_v2=True, verbose=False,
                           tedl=_tte)   # select on validation, report on test
    print(f"    Seed {_ts_seed}: F1={_tm_m['f1']:.4f}  AUC={_tm_m['roc_auc']:.4f}"
          f"  MCC={_tm_m['mcc']:.4f}  FPR@99={_tm_m.get('fpr_at_tpr99',1.):.4f}")
    _tm_seed_metrics.append(_tm_m)
    del _tm; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

_tm_summary = summarise(_tm_seed_metrics)
print(f"\nTemporal Split Results ({len(Config.FAST_SEEDS)} seeds):")
_comp_keys = ['f1', 'roc_auc', 'mcc', 'pr_auc', 'fpr_at_tpr99', 'fpr_at_tpr95']
for k in _comp_keys:
    rv = tch_summary.get(f'{k}_mean', 0)
    tv = _tm_summary.get(f'{k}_mean', 0)
    ts = _tm_summary.get(f'{k}_std', 0)
    print(f"  {k:<22} RandomSplit={rv:.4f}  Temporal={tv:.4f}±{ts:.4f}  Δ={tv-rv:+.4f}")
_tm_gap = tch_summary.get('f1_mean',0) - _tm_summary.get('f1_mean',0)
print(f"\n  Temporal-split penalty: ΔF1={_tm_gap:+.4f}")
if abs(_tm_gap) < 0.02:
    print("  ✓ Negligible gap — chronological split results are consistent with random split.")
    print("    This strongly suggests the random-split results are NOT leakage-inflated.")
elif _tm_gap > 0.05:
    print("  ⚠  Meaningful gap — random split may be overoptimistic due to temporal leakage.")
    print("    Use temporal-split numbers as the conservative headline results for paper.")
else:
    print("  Moderate gap — within acceptable bounds for a multi-dataset benchmark.")
t_res = {'temporal_f1_mean': _tm_summary.get('f1_mean',0),
          'temporal_f1_std': _tm_summary.get('f1_std',0),
          'random_f1_mean': tch_summary.get('f1_mean',0),
          'gap': _tm_gap,
          'n_seeds': len(Config.FAST_SEEDS),
          'per_seed': [{k: v for k,v in m.items() if not isinstance(v, np.ndarray)} for m in _tm_seed_metrics]}
with open(os.path.join(Config.OUT, 'temporal.json'), 'w') as f:
    json.dump(t_res, f, indent=2, default=str)
# Also update final_summary if already computed
tm_met = max(_tm_seed_metrics, key=lambda m: m['f1'])

del _ttl, _tte, Xtr_ts, ytr_ts, ctr_ts, Xte_ts, yte_ts, cte_ts
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

TEMPORAL SPLIT
  Freed main training arrays (~3.5 GB recovered)
  CICIDS-2017: train=172,473 val=24,639 test=49,278
  CIC-BCCC-NRC-TabularIoTAttacks-2024: train=40,470 val=5,781 test=11,563
  CICIoMT-2024: train=110,337 val=15,762 test=31,525
  UNSW-NB15: train=130,200 val=18,600 test=37,200
  [t_tr] 113,341 seqs (ben=64,349 56.8% | atk=48,992 43.2%)
  [t_te] 32,362 seqs (ben=18,411 56.9% | atk=13,951 43.1%)
  [t_va] 16,166 seqs (ben=9,182 56.8% | atk=6,984 43.2%)

  Training TCH-Net v2 on temporal split (3 seeds)...


  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

...

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  eval:   0%|          | 0/64 [00:00<?, ?it/s]

    Seed 456: F1=0.8849  AUC=0.9549  MCC=0.7971  FPR@99=0.7048

Temporal Split Results (3 seeds):
  f1                     RandomSplit=0.8807  Temporal=0.8831±0.0022  Δ=+0.0024
  roc_auc                RandomSplit=0.9656  Temporal=0.9629±0.0073  Δ=-0.0027
  mcc                    RandomSplit=0.7903  Temporal=0.7938±0.0055  Δ=+0.0035
  pr_auc                 RandomSplit=0.9572  Temporal=0.9542±0.0104  Δ=-0.0030
  fpr_at_tpr99           RandomSplit=0.3750  Temporal=0.4735±0.2004  Δ=+0.0985
  fpr_at_tpr95           RandomSplit=0.1840  Temporal=0.1869±0.0070  Δ=+0.0029

  Temporal-split penalty: ΔF1=-0.0024
  ✓ Negligible gap — chronological split results are consistent with random split.
    This strongly suggests the random-split results are NOT leakage-inflated.


## Values for Table 16

In [13]:
# Per-seed results and the Table 16 row (mean ± sample SD over the 3 seeds)
for _s, _m in zip(Config.FAST_SEEDS, _tm_seed_metrics):
    print(f"  seed {_s}: F1={_m['f1']:.4f}  AUC={_m['roc_auc']:.4f}  MCC={_m['mcc']:.4f}  FPR@TPR99={_m['fpr_at_tpr99']:.4f}")
print("\nTable 16 row (re-split, {} seeds):".format(len(_tm_seed_metrics)))
for _k in ['f1', 'roc_auc', 'mcc', 'fpr_at_tpr99']:
    print(f"  {_k:<13} {_tm_summary[_k+'_mean']:.4f} ± {_tm_summary[_k+'_std']:.4f}   Δ vs main = {_tm_summary[_k+'_mean'] - tch_summary[_k+'_mean']:+.4f}")


  seed 42: F1=0.8836  AUC=0.9693  MCC=0.7969  FPR@TPR99=0.3511
  seed 123: F1=0.8807  AUC=0.9647  MCC=0.7874  FPR@TPR99=0.3646
  seed 456: F1=0.8849  AUC=0.9549  MCC=0.7971  FPR@TPR99=0.7048

Table 16 row (re-split, 3 seeds):
  f1            0.8831 ± 0.0022   Δ vs main = +0.0024
  roc_auc       0.9629 ± 0.0073   Δ vs main = -0.0027
  mcc           0.7938 ± 0.0055   Δ vs main = +0.0035
  fpr_at_tpr99  0.4735 ± 0.2004   Δ vs main = +0.0985
